# Baseline Modelling — Polymarket → Gold 5-min Returns

Pipeline summary (this notebook covers steps 1-9 of the design brief):

1. **Load** the latest `gold_panel_<date>.csv` produced by the feature-engineering notebook.
2. **Load** the Bloomberg target (`GOLD USD SPOT PER OZ`, `Close`).
3. Build the target `y`: **log-return over the past 5 min, lagged −1** so that predictors at `t` predict the return realised between `t` and `t+1` (≈ +5 min).
4. Attach **auto-regressive (AR) features** to the predictors.
5. Build `X`, `y` (scikit-learn ready). Build `X_traditional` from the remaining Bloomberg indicators (loaded but not used for fitting yet).
6. **Flexible walk-forward modelling framework** with three models (Linear, LSTM, Random Forest) × four window schemes (fixed 120 / 240 / 300 + expanding). LSTM uses **batched test blocks (12 obs per retrain)** to stay tractable.
7. **Walk-forward evaluation** (train on window → predict next observation, or next 12 for LSTM).
8. **Logging**: every run appends a row to `Results/runs_log.txt` plus a detailed `.json` side-car (model spec, metrics, data hash, run timestamp).
9. **Persistence**: models are saved as `Models/<MODEL>_<WINDOW>_<DATA-DATE>.pkl|.keras`. If a matching artefact already exists and the input CSV hasn't changed, training is skipped and the model is reloaded.

> **Open questions / critical issues are collected in `NOTES_modelling_baseline.md` — please read it before interpreting results.**


## 0. Config

In [1]:

from pathlib import Path

# Folders
DATA_DIR     = Path("./Data")
MODELS_DIR   = Path("./Models")
RESULTS_DIR  = Path("./Results")
for p in (MODELS_DIR, RESULTS_DIR):
    p.mkdir(exist_ok=True, parents=True)

# File patterns
GOLD_PANEL_GLOB = "gold_panel_*.csv"          # dynamic date in file name
BLOOMBERG_XLSX_CANDIDATES = [
    DATA_DIR / "Indicators Data bloomberg.xlsx",
    DATA_DIR / "Indicators-Data-bloomberg.xlsx",
    DATA_DIR / "Indicators_Data_bloomberg.xlsx",
]
TARGET_SHEET    = "GOLD USD SPOT PER OZ"
TARGET_COL      = "Close"

def resolve_bloomberg_xlsx(candidates: list[Path], data_dir: Path) -> Path:
    for p in candidates:
        if p.exists():
            return p
    # Fallback: pick any workbook that looks like the indicators file
    wildcard = sorted(data_dir.glob("*Indicators*Data*bloomberg*.xlsx"))
    if wildcard:
        return wildcard[0]
    tried = "\n  - ".join(str(p) for p in candidates)
    raise FileNotFoundError(
        "Bloomberg workbook not found. Tried:\n"
        f"  - {tried}\n"
        "Expected something like 'Indicators Data bloomberg.xlsx' in ./Data."
    )

BLOOMBERG_XLSX = resolve_bloomberg_xlsx(BLOOMBERG_XLSX_CANDIDATES, DATA_DIR)
print(f"Using Bloomberg workbook: {BLOOMBERG_XLSX}")

# ─── Modelling ───────────────────────────────────────────────────────────────
BAR_MINUTES         = 5          # native cadence of the Bloomberg close series
RETURN_HORIZON_MIN  = 60         # ⟵ target-return horizon in MINUTES.
                                 #    60   -> 1-hour returns (current default)
                                 #    240  -> 4-hour returns (to test next)
                                 #    1440 -> next-day returns (to test later)
                                 #    5    -> old 5-min behaviour
HORIZON_STEPS       = max(1, RETURN_HORIZON_MIN // BAR_MINUTES)  # #bars ahead predicted
AR_LAGS             = [1, 2, 3, 6, 12]       # past-return lags (in 5-min bars)
AR_MA_WINDOWS       = [3, 6, 12, 36]         # rolling-mean windows on past returns
WINDOW_SCHEMES  = {                          # {label: ("fixed"|"expanding", size)}
    "fixed120":  ("fixed", 120),
    "fixed240":  ("fixed", 240),
    "fixed300":  ("fixed", 300),
    "expanding": ("expanding", 120),         # 120 = minimum training size before first prediction
}
LSTM_TEST_BLOCK = 12           # LSTM retrain cadence — predict next 12 obs per refit
LSTM_SEQ_LEN    = 12           # look-back length fed into the LSTM (≈ 1 hour)
LSTM_EPOCHS     = 8
LSTM_BATCH      = 32
RF_N_ESTIMATORS = 200
RANDOM_STATE    = 67

# ─── Regularisation knobs (tunable from here for quick iteration) ────────────
# Ridge penalty. Start around 10–100 when Polymarket features are still raw-ish;
# drop to ~1.0 once proper feature engineering / selection has been done.
RIDGE_ALPHA     = 10.0

# PCA pre-compression applied to the Polymarket block ONLY (AR features are
# kept raw, they are already in return-space and low-dim).
# Set PCA_N_COMPONENTS = 0 (or None) to disable PCA entirely.
#   typical range: 20–40 while Polymarket dim is high;
#   after feature engineering you can probably drop it to 0.
PCA_N_COMPONENTS = None

# ─── Traditional indicators — separate PCA and feature engineering controls ──
# Independent PCA for the Bloomberg traditional block: typically 3–8 components
# (one per macro factor: rates, equities, FX, commodities, vol, etc.).
# Set to 0 to disable PCA on the traditional block entirely.
PCA_N_COMPONENTS_TRADITIONAL = None

# Forward-fill cap when resampling daily Bloomberg indicators to the 5-min gold
# clock.  None = carry indefinitely (standard for Bloomberg daily data);
# set an integer to cap stale carries (e.g., 288 = ~1 trading day on a 5-min grid).
TRADITIONAL_MAX_FFILL_BARS   = None

# Add a staleness counter feature for each traditional indicator:
# tells the model how many 5-min bars have elapsed since the last real tick.
# Useful because daily series are stale for ~288 bars between updates.
TRADITIONAL_USE_STALENESS    = True

# Drop raw price-level columns after stationarising (default True).
# They are collinear with the diff/log-return columns and can confuse Ridge.
TRADITIONAL_DROP_PRICE_LEVELS = True

# AR features of the gold log-return appended to the traditional predictor matrix.
# Must mirror the full AR feature set used for the polymarket block so that both
# sets of models see the same autoregressive information:
#   - 5 lagged returns  (AR_LAGS)
#   - 4 rolling means   (AR_MA_WINDOWS)
#   - 2 rolling stds    ([12, 36])
# These are passed separately to build_X_traditional via ar_lags / ar_ma_windows /
# ar_vol_windows; keeping them as explicit config allows independent tuning later.
TRADITIONAL_AR_LAGS       = [1, 2, 3, 6, 12]   # must match AR_LAGS
# ar_ma_windows and ar_vol_windows for the trad block are passed as AR_MA_WINDOWS
# and [12, 36] directly in Cell 5 (they share the same config as the poly block).

# ─── Feature engineering checkmark (variance prefilter gate) ─────────────────
# NOTE: the variance pre-filter in Cell 4.1 is a stop-gap that keeps LSTM/Ridge
# tractable while Polymarket features are still raw and high-dimensional. Once
# proper feature engineering AND selection have been done upstream (on the
# polymarket panel), the prefilter is no longer needed and can be skipped:
#   - FEATURE_ENGINEERING_DONE = False  -> prefilter ON  (keep top-N by variance)
#   - FEATURE_ENGINEERING_DONE = True   -> prefilter OFF (use all columns)
FEATURE_ENGINEERING_DONE = False

MAX_FEATURES_PREFILTER = 150   # crude variance prefilter (only used if FE not done)

# ─── Daily gap handling (Bloomberg ~23:00 break, see Cell 10) ────────────────
# Adds an `is_post_break` dummy AND a `poly_movement_during_break` feature
# (approach 2 + 3 combined — see the PDF diagnosis). Turn off to revert.
HANDLE_DAILY_GAP = True
GAP_THRESHOLD    = "60min"       # a bar-to-bar step > this counts as a session break

# ─── Feature-engineering input mode ─────────────────────────────────────────
# FE_INPUT_MODE controls which polymarket panel file is loaded and whether
# in-window PLS/Lasso preprocessing is applied inside model wrappers.
#
#   "raw_panel"    — input is the filtered raw polymarket panel produced by the
#                    PLS Feature Engineering notebook (polymarket_panel_filtered_
#                    <TIMESTAMP>.csv).  PLS/Lasso are applied as per-window
#                    preprocessing inside the PLS-variant model wrappers.
#
#   "preprocessed" — input is already a low-dimensional engineered representation
#                    (e.g., from a separate PCA-based feature engineering script,
#                    polymarket_panel_preprocessed_<TIMESTAMP>.csv).  PLS
#                    preprocessing inside models is SKIPPED to avoid double-
#                    reducing an already-compressed panel.
FE_INPUT_MODE = "raw_panel"

# Sensitivity-check controls for alternative PLS component counts.
RUN_PLS_SENSITIVITY_SWEEP = False
PLS_SENSITIVITY_GRID      = [3, 5, 8, 10]

# ─── Dataset run toggles ─────────────────────────────────────────────────────
# Set any flag to False to skip that dataset entirely (no training, no logging).
# This lets you run only the subset of models you care about in a given session.
#
# Datasets and where they are consumed:
#   RUN_POLY           → "poly"           Cell 8   Polymarket features + AR lags
#   RUN_TRAD           → "trad"           Cell 8   Bloomberg traditional features + AR lags
#   RUN_COMBINED       → "poly+trad"      Cell 15  All features combined (dual-PCA)
#   RUN_POLY_ONLY      → "poly_only"      Cell 16  Polymarket features ONLY (no AR)
#   RUN_BLOOMBERG_ONLY → "bloomberg_only" Cell 16  Bloomberg features ONLY (no AR)
#   RUN_AR_ONLY        → "ar_only"        Cell 16  AR gold log-return features ONLY
RUN_POLY           = True
RUN_TRAD           = True
RUN_COMBINED       = True
RUN_POLY_ONLY      = True
RUN_BLOOMBERG_ONLY = True
RUN_AR_ONLY        = True

# Evaluation toggles
FORCE_RETRAIN   = False        # set True to ignore cached models
DRY_RUN_ROWS    = None         # e.g. 1000 to debug on a slice; None = full


Using Bloomberg workbook: Data\Indicators Data bloomberg.xlsx


In [2]:
# %% ── CELL 0.1 : LOAD PLS n_components FROM FE ARTEFACT ─────────────────────
import json

PLS_SELECTION_LOG_DIR = Path("Feature selection log")
PLS_SELECTION_ARTEFACT_PATH = PLS_SELECTION_LOG_DIR / "pls_n_components_latest.json"

if not PLS_SELECTION_ARTEFACT_PATH.exists():
    raise FileNotFoundError(
        "Missing PLS n_components artefact at "
        f"{PLS_SELECTION_ARTEFACT_PATH.resolve()}. "
        "Run 'PLS - Feature Engineering SQL polymarket database.ipynb' first "
        "to generate Feature selection log/pls_n_components_latest.json."
    )

PLS_SELECTION_METADATA = json.loads(
    PLS_SELECTION_ARTEFACT_PATH.read_text(encoding="utf-8")
)
_loaded_pls_n_components = PLS_SELECTION_METADATA.get("n_components")
if (
    isinstance(_loaded_pls_n_components, bool)
    or not isinstance(_loaded_pls_n_components, int)
    or _loaded_pls_n_components < 1
):
    raise ValueError(
        "Invalid PLS selection artefact: expected an integer n_components >= 1, "
        f"got {_loaded_pls_n_components!r}."
    )

PLS_N_COMPONENTS = _loaded_pls_n_components
print(f"Loaded PLS_N_COMPONENTS   : {PLS_N_COMPONENTS}")
print(f"Artefact path             : {PLS_SELECTION_ARTEFACT_PATH}")
print(
    "Artefact generated_at_utc : ",
    PLS_SELECTION_METADATA.get("generated_at_utc", "<missing>")
)
print(
    "Artefact grid_searched    : ",
    PLS_SELECTION_METADATA.get("grid_searched", "<missing>")
)

Loaded PLS_N_COMPONENTS   : 3
Artefact path             : Feature selection log\pls_n_components_latest.json
Artefact generated_at_utc :  2026-05-13T18:39:42Z
Artefact grid_searched    :  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]


## 1. Dependencies

In [3]:
# %% ── CELL 1 : DEPENDENCIES ─────────────────────────────────────────────────
import os, re, json, hashlib, joblib, warnings, datetime as dt, time
import numpy as np
import pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

from sklearn.linear_model import Ridge
from sklearn.ensemble    import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics     import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import VarianceThreshold

# TensorFlow / Keras (LSTM) — imported lazily so the notebook still runs if TF isn't installed
_TF_AVAILABLE = True
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential, load_model
    from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
    from tensorflow.keras.callbacks import EarlyStopping
    tf.random.set_seed(RANDOM_STATE)
except Exception as _tf_err:
    _TF_AVAILABLE = False
    print("⚠️  TensorFlow not importable — LSTM runs will be skipped.", _tf_err)

np.random.seed(RANDOM_STATE)
print("✅ Dependencies loaded. TF:", _TF_AVAILABLE)


✅ Dependencies loaded. TF: True


## 1.1 Traditional predictor helpers

Functions for building the Bloomberg traditional predictor matrix:
- **`make_stationary_features`** — log-diff for positive price series, simple diff for others.
- **`build_X_traditional`** — reindexes each Bloomberg sheet to the gold 5-min clock, forward-fills, stationarises, optionally adds staleness counters, then aligns to the modelling index.
- **`fit_transform_pca_fold`** — fold-safe PCA (fit on train, apply to test) used inside the walk-forward loop for the traditional block.

In [4]:
# %% ── CELL 1.1 : TRADITIONAL PREDICTOR HELPERS ─────────────────────────────

def make_stationary_features(df: pd.DataFrame, exclude_cols=None) -> pd.DataFrame:
    """Convert each column to a stationary series:
    - strictly positive series  → log-diff  (captures percentage moves)
    - other numeric series      → simple diff (rates, spreads, indices that cross 0)
    Columns in `exclude_cols` (e.g. staleness counters) are passed through as-is.
    """
    exclude_cols = set(exclude_cols or [])
    out = pd.DataFrame(index=df.index)
    for col in df.columns:
        s = pd.to_numeric(df[col], errors="coerce")
        if col in exclude_cols:
            out[col] = s
            continue
        strictly_positive = (s.dropna() > 0).all()
        if strictly_positive:
            out[f"{col}_logret"] = np.log(s).diff()
        else:
            out[f"{col}_diff"] = s.diff()
    return out


def build_X_traditional(
    bb_dict: dict,
    target_sheet: str,
    master_index: pd.DatetimeIndex,
    align_index: pd.DatetimeIndex,
    max_ffill_bars=None,
    use_staleness: bool = True,
    ar_lags=(1, 2, 3, 6, 12),
    ar_ma_windows=(3, 6, 12, 36),
    ar_vol_windows=(12, 36),
    logret_bar: pd.Series = None,
) -> pd.DataFrame:
    """Build a stationary traditional predictor matrix aligned to `align_index`.

    Pipeline per non-target Bloomberg sheet:
    1. Reindex the close column to `master_index` (5-min gold clock).
    2. Optionally compute a staleness counter (bars since last real tick).
    3. Forward-fill up to `max_ffill_bars`.
    4. Stationarise: log-diff (positive series) or simple diff (other).
    5. Append AR lags, rolling means, and rolling stds of the gold log-return
       (same full AR feature set as build_ar_features for the polymarket block).
    6. Reindex everything to `align_index` (= X.index, valid polymarket rows).

    Reuses the already-loaded `bb` dict — no extra workbook I/O.
    """
    import re as _re
    price_cols_raw = {}   # {feature_name: forward-filled Series}
    stale_cols_raw = {}   # {stale_col_name: counter Series}

    for sheet, df in bb_dict.items():
        if sheet == target_sheet:
            continue
        if "close" not in df.columns:
            continue

        feature_name = _re.sub(r"[^0-9a-zA-Z]+", "_", sheet).strip("_").lower()
        s_raw = df["close"].sort_index()
        s_raw = s_raw[~s_raw.index.duplicated(keep="last")]

        # Align to 5-min master index
        s_aligned = s_raw.reindex(master_index)

        if use_staleness:
            # Count bars since the last real (non-NaN) tick
            is_new_tick       = s_aligned.notna()
            real_tick_groups  = is_new_tick.cumsum()
            stale_count       = (~is_new_tick).groupby(real_tick_groups).cumsum().astype(int)
            stale_cols_raw[f"{feature_name}_stale"] = stale_count

        s_filled = s_aligned.ffill(limit=max_ffill_bars)
        price_cols_raw[feature_name] = s_filled

    if not price_cols_raw:
        return pd.DataFrame(index=align_index)

    price_df   = pd.DataFrame(price_cols_raw,  index=master_index)
    stale_df   = pd.DataFrame(stale_cols_raw,  index=master_index)

    # Stationarise price-level columns
    stationary = make_stationary_features(price_df)

    # Merge staleness counters back (they stay in levels — already stationary-ish)
    if use_staleness and not stale_df.empty:
        stationary = stationary.join(stale_df, how="left")

    # Add the full AR feature set for the gold log-return — mirrors build_ar_features
    # used for the polymarket block (5 lagged returns, 4 rolling means, 2 rolling stds).
    if logret_bar is not None:
        logret_aligned = logret_bar.reindex(master_index)
        for lag in ar_lags:
            stationary[f"trad_gold_lag{lag}"] = logret_aligned.shift(lag)
        for w in ar_ma_windows:
            stationary[f"trad_gold_ma{w}"] = logret_aligned.shift(1).rolling(w).mean()
        for w in ar_vol_windows:
            stationary[f"trad_gold_std{w}"] = logret_aligned.shift(1).rolling(w).std()

    # Final alignment to the modelling index
    stationary = stationary.reindex(align_index)
    stationary = stationary.replace([np.inf, -np.inf], np.nan)
    stationary = stationary.ffill().fillna(0.0)   # 0.0 for series-start NaN

    return stationary


def fit_transform_pca_fold(
    X_train: np.ndarray,
    X_test:  np.ndarray,
    n_components: int,
    prefix: str,
    random_state: int = RANDOM_STATE,
):
    """Fold-safe PCA: fit only on `X_train`, transform both splits.
    Returns raw numpy arrays (not DataFrames) so they drop straight into the
    existing ScaledRegressor / LSTM flow.
    Returns (X_train_out, X_test_out, pca_obj); pca_obj is None if PCA is skipped.
    """
    if (not n_components) or (n_components <= 0) or (X_train.shape[1] <= n_components):
        return X_train, X_test, None
    n_comp = min(n_components, X_train.shape[1], X_train.shape[0])
    pca = PCA(n_components=n_comp, random_state=random_state)
    X_train_out = pca.fit_transform(X_train)
    X_test_out  = pca.transform(X_test)
    return X_train_out, X_test_out, pca

print("✅ Traditional predictor helpers loaded.")


✅ Traditional predictor helpers loaded.


## 2. Locate the latest gold panel CSV

The feature-engineering pipeline writes `gold_panel_<YYYY-MM-DD>.csv` to `./Data`. Here we pick the file with the most recent date in its name (NOT file mtime — mtime can be misleading if the file was merely copied).

In [5]:

# %% ── CELL 2 : LOAD LATEST GOLD PANEL ───────────────────────────────────────
_date_rx = re.compile(r"gold_panel_(\d{4}-\d{2}-\d{2})\.csv$")

def find_latest_panel(folder: Path) -> tuple[Path, str]:
    candidates = []
    for p in folder.glob(GOLD_PANEL_GLOB):
        m = _date_rx.search(p.name)
        if m:
            candidates.append((m.group(1), p))
    if not candidates:
        raise FileNotFoundError(f"No files matching {GOLD_PANEL_GLOB} in {folder}")
    candidates.sort(key=lambda t: t[0])   # lexicographic sort works for ISO dates
    return candidates[-1][1], candidates[-1][0]

# ── Select panel file based on FE_INPUT_MODE ──────────────────────────────────
if FE_INPUT_MODE == "raw_panel":
    # Primary output of the PLS Feature Engineering notebook after all filters.
    # AR features are NOT included — this notebook builds them from scratch.
    _poly_rx   = re.compile(r"polymarket_panel_filtered_(\d{4}-\d{2}-\d{2})\.csv$")
    _poly_glob = "polymarket_panel_filtered_*.csv"

    def _find_latest_filtered_panel(folder: Path) -> tuple[Path, str]:
        candidates = []
        for p in folder.glob(_poly_glob):
            m = _poly_rx.search(p.name)
            if m:
                candidates.append((m.group(1), p))
        if not candidates:
            raise FileNotFoundError(
                f"No polymarket_panel_filtered_*.csv found in {folder}.\n"
                "Run the PLS Feature Engineering notebook first, or set "
                "FE_INPUT_MODE='preprocessed' for the PCA-based alternative."
            )
        candidates.sort(key=lambda t: t[0])
        return candidates[-1][1], candidates[-1][0]

    panel_path, panel_date = _find_latest_filtered_panel(DATA_DIR)

elif FE_INPUT_MODE == "preprocessed":
    # Placeholder path for a PCA-based pre-engineered panel produced by a
    # parallel feature engineering script.  Replace the filename once that
    # script generates its output.
    panel_path = DATA_DIR / "polymarket_panel_preprocessed_YYYY-MM-DD.csv"  # TODO: update
    panel_date = "unknown"
    if not panel_path.exists():
        raise FileNotFoundError(
            f"Preprocessed panel not found: {panel_path}\n"
            "Set FE_INPUT_MODE='raw_panel' or place the preprocessed CSV at the above path."
        )

else:
    raise ValueError(f"Unknown FE_INPUT_MODE: {FE_INPUT_MODE!r}")

print(f"FE_INPUT_MODE  : {FE_INPUT_MODE}")
print(f"Latest panel   : {panel_path.name}  (data date = {panel_date})")

X_raw = pd.read_csv(panel_path, parse_dates=["scraped_at"]).set_index("scraped_at").sort_index()
if DRY_RUN_ROWS:
    X_raw = X_raw.iloc[-DRY_RUN_ROWS:]
print(f"X_raw shape    : {X_raw.shape}   range: {X_raw.index.min()} → {X_raw.index.max()}")



FE_INPUT_MODE  : raw_panel
Latest panel   : polymarket_panel_filtered_2026-05-07.csv  (data date = 2026-05-07)
X_raw shape    : (7171, 1500)   range: 2026-04-01 00:00:00 → 2026-05-07 22:00:00


## 3. Load Bloomberg indicators — build target and X_traditional

The Bloomberg workbook has one sheet per indicator, each with 5 columns (`Date, Open, High, Low, Close`). We keep the **Close** column of every sheet. `GOLD USD SPOT PER OZ` becomes the target; the rest go into `X_traditional`.

In [6]:
# %% ── CELL 3.0 : ENSURE OPENPYXL ───────────────────────────────────────────
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("openpyxl") is None:
    print("openpyxl not found. Installing via pip...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "openpyxl"])
else:
    print("openpyxl is already installed.")


openpyxl is already installed.


In [7]:
# %% ── CELL 3 : LOAD BLOOMBERG SHEETS ────────────────────────────────────────
def load_bloomberg(xlsx_path: Path) -> dict[str, pd.DataFrame]:
    '''Load every Bloomberg sheet that has a `Date` column.
       Some sheets use `Close`, some use `Last Price` — we normalise both to `close`.'''
    xl = pd.ExcelFile(xlsx_path)
    out = {}
    for sheet in xl.sheet_names:
        df = pd.read_excel(xlsx_path, sheet_name=sheet, header=0)
        if df.empty or "Date" not in df.columns:
            continue
        df = df.rename(columns={c: c.strip() for c in df.columns})
        df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
        df = df.dropna(subset=["Date"]).set_index("Date").sort_index()
        for c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
        # normalise the price column name
        if "Close" in df.columns:
            df = df.rename(columns={"Close": "close"})
        elif "Last Price" in df.columns:
            df = df.rename(columns={"Last Price": "close"})
        else:
            numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
            if not numeric_cols:
                continue
            df = df.rename(columns={numeric_cols[-1]: "close"})
        out[sheet] = df
    return out

bb = load_bloomberg(BLOOMBERG_XLSX)
print("Bloomberg sheets loaded:", list(bb.keys()))
gold = bb[TARGET_SHEET][["close"]].rename(columns={"close": "gold_close"})
print("Gold close range:", gold.index.min(), "→", gold.index.max(), "| rows:", len(gold))


Bloomberg sheets loaded: ['GOLD USD SPOT PER OZ', 'CRUDE OIL (WTI) FUTURES PRICE', 'CRUDE OIL (BRENT) FUTURES PRICE', 'S&P 500 Index', 'USD Index', 'VIX Index', 'USGG10YR Index', 'EURUSD', 'GC1 Comdty', 'GLD US Equity']
Gold close range: 2026-04-01 00:00:00 → 2026-05-08 22:55:00 | rows: 7459


### 3.1 Build the target `y`

We want a **regression** target on **future returns**, not prices.

\[
y_t = \log\!\left(\frac{P_{t+1}}{P_t}\right)
\]

so that predictors available at time `t` forecast the realised 5-min log-return over `[t, t+1]`. Equivalently: take the 5-min log-return series and shift it by `-1`.

**Why log-returns vs simple returns?** Additivity across time (log-returns sum), symmetry, and far more stable numerics for small moves — standard in high-frequency finance.

In [8]:
# %% ── CELL 3.1 : BUILD y (log-returns over RETURN_HORIZON_MIN, lagged -HORIZON_STEPS) ───
# ─── FIX (from Operation Fixing Terrible Performance, point 5) ───────────────
# We now use a CONFIGURABLE horizon. The per-bar log-return is still computed
# at the native 5-min cadence (it's what AR features need), but the TARGET is
# the log-return over HORIZON_STEPS bars ahead:
#     y_t = log(P_{t+H}) - log(P_t)
# where H = HORIZON_STEPS = RETURN_HORIZON_MIN / BAR_MINUTES.
#
# Switching RETURN_HORIZON_MIN in the config cell is the only change needed
# to move from 1h -> 4h -> next-day returns.
# Bloomberg sheets list rows newest-first; sort_index() already fixed that.
gold["logret_bar"]     = np.log(gold["gold_close"]).diff()                    # per-bar (5-min) log-return
gold["logret_horizon"] = (np.log(gold["gold_close"])
                          - np.log(gold["gold_close"]).shift(HORIZON_STEPS))  # return realised over [t-H, t]
gold["y"]              = gold["logret_horizon"].shift(-HORIZON_STEPS)         # shift forward = realised over [t, t+H]
# Backwards-compat alias (some downstream cells still reference `logret_5m`).
gold["logret_5m"]      = gold["logret_bar"]
# (gold.index = t. gold["y"].loc[t] = log(P_{t+H}) - log(P_t), i.e. the H-bar-ahead return.)
print(f"Target horizon: {RETURN_HORIZON_MIN} min  ({HORIZON_STEPS} bars of {BAR_MINUTES} min each)")
print(gold[["gold_close","logret_bar","logret_horizon","y"]].tail(6))


Target horizon: 15 min  (3 bars of 5 min each)
                     gold_close  logret_bar  logret_horizon         y
Date                                                                 
2026-05-08 22:30:00     4717.14   -0.000049        0.000339 -0.000339
2026-05-08 22:35:00     4717.88    0.000157        0.000184 -0.000581
2026-05-08 22:40:00     4716.74   -0.000242       -0.000134 -0.000316
2026-05-08 22:45:00     4715.54   -0.000254       -0.000339       NaN
2026-05-08 22:50:00     4715.14   -0.000085       -0.000581       NaN
2026-05-08 22:55:00     4715.25    0.000023       -0.000316       NaN


### 3.2 Auto-regressive (AR) features

**Design choice — rationale:**
Gold returns at 5-min cadence exhibit small but exploitable **short-term momentum / mean-reversion** and strong **volatility clustering**. Any non-trivial baseline must give the model access to its own past, otherwise we're asking Polymarket features alone to beat a signal that's already in the price path.

I'm adding three groups of AR features, all computed on `logret_5m` (which is *already observed* at time `t`, so no look-ahead):

1. **Lagged returns** at 1, 2, 3, 6, 12 bars (5, 10, 15, 30, 60 min). Captures raw momentum.
2. **Rolling means of past returns** over 3, 6, 12, 36 bars. Same idea as the lectures' moving-average baseline; a denoised direction estimate.
3. **Rolling std (realised vol proxy)** over 12 and 36 bars. Lets tree models condition on the current vol regime — classic HAR-RV intuition.

All features use only information up to and including `t`, so there is **no leakage**. They are attached to the `X` predictor matrix after timestamp alignment.

In [9]:
# %% ── CELL 3.2 : AR FEATURES ────────────────────────────────────────────────
# AR features are always built on the NATIVE 5-min log-returns (logret_bar).
# They capture short-term momentum / vol-clustering at the bar cadence; this
# is independent of the target horizon.
def build_ar_features(logret: pd.Series,
                      lags=AR_LAGS,
                      ma_windows=AR_MA_WINDOWS,
                      vol_windows=(12, 36)) -> pd.DataFrame:
    feats = {}
    for L in lags:
        feats[f"ar_ret_lag{L}"] = logret.shift(L)
    for W in ma_windows:
        feats[f"ar_ret_ma{W}"]  = logret.shift(1).rolling(W).mean()
    for W in vol_windows:
        feats[f"ar_ret_std{W}"] = logret.shift(1).rolling(W).std()
    return pd.DataFrame(feats)

ar_feats = build_ar_features(gold["logret_bar"])
print("AR feature matrix shape:", ar_feats.shape)
print(ar_feats.tail(3))


AR feature matrix shape: (7459, 11)
                     ar_ret_lag1  ar_ret_lag2  ar_ret_lag3  ar_ret_lag6  \
Date                                                                      
2026-05-08 22:45:00    -0.000242     0.000157    -0.000049    -0.000738   
2026-05-08 22:50:00    -0.000254    -0.000242     0.000157     0.000312   
2026-05-08 22:55:00    -0.000085    -0.000254    -0.000242     0.000076   

                     ar_ret_lag12  ar_ret_ma3  ar_ret_ma6  ar_ret_ma12  \
Date                                                                     
2026-05-08 22:45:00     -0.000205   -0.000045   -0.000081    -0.000153   
2026-05-08 22:50:00      0.000068   -0.000113    0.000000    -0.000157   
2026-05-08 22:55:00     -0.000102   -0.000194   -0.000066    -0.000170   

                     ar_ret_ma36  ar_ret_std12  ar_ret_std36  
Date                                                          
2026-05-08 22:45:00    -0.000024      0.000302      0.000345  
2026-05-08 22:50:00    -0.00

## 4. Align Polymarket features to Bloomberg's 5-min grid, assemble `X` and `y`

Polymarket is scraped every ~5 min but the timestamps are **not on the clock** (14:15:52, 14:21:07, …), whereas Bloomberg prices sit exactly on `:00/:05/:10/…`. To merge them I use `merge_asof` with a 5-min tolerance and `direction="backward"`: for every Bloomberg bar, I grab the **latest** Polymarket snapshot strictly at or before that bar. This is the only look-ahead-safe way to align the two clocks.

After alignment we drop rows with no target (the last bar and any pre-warmup AR rows).

In [10]:
# %% ── CELL 4 : ALIGN + BUILD X, y ───────────────────────────────────────────
# 1) Restrict Bloomberg to the period covered by the panel
start, end = X_raw.index.min().floor("5min"), X_raw.index.max().ceil("5min")
gold_aligned = gold.loc[(gold.index >= start) & (gold.index <= end)].copy()
print(f"Bloomberg bars in overlap window: {len(gold_aligned)}")

# 2) merge_asof: for each 5-min Bloomberg bar, pull the latest polymarket snapshot ≤ bar
X_raw_sorted = X_raw.sort_index()
merged = pd.merge_asof(
    left      = gold_aligned.reset_index().rename(columns={"Date":"ts"}),
    right     = X_raw_sorted.reset_index().rename(columns={"scraped_at":"ts"}),
    on        = "ts",
    direction = "backward",
    tolerance = pd.Timedelta(f"{BAR_MINUTES}min"),
).set_index("ts")

# 3) Pull in AR features (index already Bloomberg bars)
merged = merged.join(ar_feats, how="left")

# 4) Separate into X and y
y = merged["y"]
polymarket_cols = [c for c in X_raw.columns]
ar_cols         = list(ar_feats.columns)
feature_cols    = polymarket_cols + ar_cols
X = merged[feature_cols].copy()

# 5) Drop rows where y or AR features are undefined
valid = y.notna() & merged[ar_cols].notna().all(axis=1)
X, y = X.loc[valid], y.loc[valid]

# 6) Column-level NA handling: forward-fill within polymarket cols (stale quote is the truth),
#    then fill remaining with 0 (market that didn't exist yet).
X[polymarket_cols] = X[polymarket_cols].ffill().fillna(0.0)

# NOTE: Stationarity transformation (differencing / log-differencing) is now applied
# upstream in the Feature Engineering notebook before the gold_panel CSV is written.
# The X_raw columns loaded here are therefore already stationary — no further
# differencing is needed at this stage.

print(f"Final X shape: {X.shape}")
print(f"Final y shape: {y.shape}")
print(f"y sample stats: mean={y.mean():.2e}  std={y.std():.2e}  min={y.min():.2e}  max={y.max():.2e}")


Bloomberg bars in overlap window: 7171
Final X shape: (7134, 1511)
Final y shape: (7134,)
y sample stats: mean=6.99e-07  std=1.61e-03  min=-1.76e-02  max=1.02e-02


### 4.1 Variance pre-filter (conditional) + daily-gap features

Two things happen in this cell:

1. **Variance pre-filter (conditional)** — when `FEATURE_ENGINEERING_DONE = False` in the config cell, we drop near-constants and keep the top-`MAX_FEATURES_PREFILTER` columns by variance (plus all AR features). Once feature engineering / selection has been done upstream, flip the flag to `True` and this filter is skipped. Variance ≠ relevance — this is a stop-gap, not real selection.
2. **Daily-gap features (Approach 2 + 3)** — `is_post_break` is a regime dummy; `poly_movement_during_break` sums `|ΔPolymarket|` across the overnight break and captures the information accumulated while the gold feed was closed. Toggle with `HANDLE_DAILY_GAP`. The other approaches stay in Cell 10 (commented out) so you can revert.


In [11]:

# %% ── CELL 4.1 : VARIANCE PREFILTER + DAILY-GAP FEATURES ──────────────────
# ─── FIX (Operation Fixing Terrible Performance, point 3) ────────────────────
# The variance pre-filter is a STOP-GAP for the period when Polymarket features
# are still raw and high-dimensional. Once proper feature engineering and
# selection have been done upstream (flip FEATURE_ENGINEERING_DONE = True in
# the config cell), the filter is skipped and every surviving column is kept.
# NOTE: variance ≠ relevance — a slowly-moving probability that just crossed
#       a threshold can be very informative. Keeping this only until FE is in.
if FEATURE_ENGINEERING_DONE:
    print("🆗 FEATURE_ENGINEERING_DONE=True -> skipping variance prefilter.")
    X_all      = X.copy()
    # AR features are guaranteed to already be in X — no-op.
    print(f"X shape (no prefilter): {X.shape}")
else:
    vt = VarianceThreshold(threshold=1e-8)
    vt.fit(X.values)
    kept_mask  = vt.get_support()
    kept_cols  = X.columns[kept_mask]
    print(f"After VarianceThreshold: {len(kept_cols)} / {X.shape[1]} columns survive.")

    # Keep the top-N by variance (heuristic only — cheap to change later)
    variances    = X[kept_cols].var().sort_values(ascending=False)
    top_cols     = variances.head(MAX_FEATURES_PREFILTER).index.tolist()
    # Always keep AR features regardless of variance ranking
    for c in ar_cols:
        if c in X.columns and c not in top_cols:
            top_cols.append(c)

    X_all = X.copy()                 # untouched full matrix (for reference)
    X     = X[top_cols]
    print(f"X shape after prefilter : {X.shape}")

# Re-sync the polymarket column list to whatever actually survived in X.
# (Used downstream for PCA / gap features.)
polymarket_cols_kept = [c for c in X.columns if c not in ar_cols]
print(f"  polymarket cols kept: {len(polymarket_cols_kept)}  |  AR cols: {len(ar_cols)}")

# ─── FIX (Operation Fixing Terrible Performance, point 4) ────────────────────
# Bloomberg's gold feed has a ~1h daily break starting ~23:00. The first bar
# after re-open is actually a ~65-minute return carrying ALL the information
# that accumulated during the break — dropping it wastes signal. So we DO NOT
# mask it (Approach 1). Instead we combine Approach 2 + 3:
#   (2) is_post_break dummy -> lets the model learn the different regime;
#   (3) poly_movement_during_break = sum of |ΔPolymarket| during the gap
#       -> the accumulated-information signal.
# Approach 1 and Approach 4 remain in Cell 10 (commented out) so we can revert.
# This block runs BEFORE the training grid (unlike the original Cell 10, which
# was positioned after the grid and therefore had no effect on training).
if HANDLE_DAILY_GAP:
    _gap_step      = X.index.to_series().diff()
    _is_post_break = (_gap_step > pd.Timedelta(GAP_THRESHOLD)).astype(int)

    # (2) regime dummy
    X["is_post_break"] = _is_post_break.values

    # (3) polymarket movement accumulated during the gap
    # NOTE: X_raw below is the UN-FILTERED, pre-variance-prefilter Polymarket panel
    # (every Polymarket tick at original resolution). This is intentional: we want
    # to capture ALL Polymarket activity during the gap, not just the columns that
    # survived the variance filter. X_raw remains unchanged throughout Cell 4.1.
    _poly_abs_delta = X_raw.sort_index().diff().abs().sum(axis=1)
    _gap_feature    = pd.Series(0.0, index=X.index)
    _X_times = X.index.to_list()
    for _i in range(1, len(_X_times)):
        _t_prev, _t_curr = _X_times[_i-1], _X_times[_i]
        if (_t_curr - _t_prev) > pd.Timedelta(GAP_THRESHOLD):
            _mask = (_poly_abs_delta.index > _t_prev) & (_poly_abs_delta.index <= _t_curr)
            _gap_feature.iloc[_i] = float(_poly_abs_delta.loc[_mask].sum())
    X["poly_movement_during_break"] = _gap_feature.values

    # Both new features are behavioural / AR-like — keep them OUT of the PCA
    # block so Ridge sees them raw. We extend ar_cols to reflect that.
    ar_cols = list(ar_cols) + ["is_post_break", "poly_movement_during_break"]
    print(f"✅ Daily-gap handling ON.  post-break bars: {int(_is_post_break.sum())}  |  "
          f"non-zero 'poly_movement_during_break': {int((_gap_feature != 0).sum())}")
    print(f"  new X shape: {X.shape}")
else:
    print("ℹ️  Daily-gap handling OFF (HANDLE_DAILY_GAP=False).")

# ── Column index arrays for ColumnTransformer routing ─────────────────────────
# POLY_COLS: integer indices of polymarket columns in X (go through PLS).
# AR_COLS:   integer indices of AR/gap columns in X (pass through PLS unchanged).
# These are derived from the actual final X, not hard-coded, so they survive
# the variance prefilter and gap-feature additions above.
_ar_col_set = set(ar_cols)
POLY_COLS = [i for i, c in enumerate(X.columns) if c not in _ar_col_set]
AR_COLS   = [i for i, c in enumerate(X.columns) if c in _ar_col_set]
print(f"POLY_COLS: {len(POLY_COLS)} columns  (will go through PLS in PLS-variant models)")
print(f"AR_COLS:   {len(AR_COLS)} columns  (pass through PLS unchanged)")



After VarianceThreshold: 517 / 1511 columns survive.
X shape after prefilter : (7134, 161)
  polymarket cols kept: 150  |  AR cols: 11
✅ Daily-gap handling ON.  post-break bars: 19  |  non-zero 'poly_movement_during_break': 15
  new X shape: (7134, 163)
POLY_COLS: 150 columns  (will go through PLS in PLS-variant models)
AR_COLS:   13 columns  (pass through PLS unchanged)


## 5. Build `X_traditional` (NOT used in modelling yet — stored for later)

In [12]:
# %% ── CELL 5 : X_traditional ───────────────────────────────────────────────
# ─── OPERATION ALIGN TRADITIONAL PREDICTORS ──────────────────────────────────
# Replaces the previous simplistic reindex+log-diff approach with a
# stationarity-safe, gold-clock-aligned build that:
#   (a) reindexes each Bloomberg sheet to the 5-min gold clock (master_index)
#       before differencing — so daily series show non-zero returns only at
#       their daily update bar, and 0 at all other 5-min bars;
#   (b) adds optional staleness counters (bars since last real tick);
#   (c) appends AR lags of the gold log-return (matching the polymarket block);
#   (d) aligns the final matrix to X.index (same valid rows as polymarket X).
# Reuses the `bb` dict already loaded in Cell 3 — no extra workbook I/O.
# ─────────────────────────────────────────────────────────────────────────────

# `gold.index` is the full 5-min Bloomberg gold clock — use it as master.
# `X.index`   is the valid modelling rows (polymarket × Bloomberg overlap).
X_traditional = build_X_traditional(
    bb_dict        = bb,
    target_sheet   = TARGET_SHEET,
    master_index   = gold.index,          # full 5-min Bloomberg gold clock
    align_index    = X.index,             # restrict to valid modelling rows
    max_ffill_bars = TRADITIONAL_MAX_FFILL_BARS,
    use_staleness  = TRADITIONAL_USE_STALENESS,
    ar_lags        = TRADITIONAL_AR_LAGS,
    ar_ma_windows  = AR_MA_WINDOWS,       # same rolling-mean windows as poly block
    ar_vol_windows = [12, 36],            # same rolling-std windows as poly block
    logret_bar     = gold["logret_bar"],
)

# y is the same target as the polymarket block (same gold returns, same index)
y_traditional = y.loc[X_traditional.index]

# ─── Daily-gap features for the traditional block ────────────────────────────
# Same methodology as Cell 4.1 for the polymarket block: we add two dummy
# features to X_traditional so that trad models can also learn the overnight
# break regime and the price information accumulated during the gap.
if HANDLE_DAILY_GAP:
    _trad_gap_step      = X_traditional.index.to_series().diff()
    _trad_is_post_break = (_trad_gap_step > pd.Timedelta(GAP_THRESHOLD)).astype(int)

    # (2) regime dummy — mirrors is_post_break in the poly block
    X_traditional["is_post_break"] = _trad_is_post_break.values

    # (3) traditional movement accumulated during the gap
    # NOTE: gold["logret_bar"] below is the UN-FILTERED, pre-alignment gold
    # log-return series (full Bloomberg gold clock). This is intentional: we
    # want to capture ALL gold price movement during the gap window, not just
    # the rows that survived alignment to X.index. The series is re-aligned
    # to X_traditional.index via reindex so the loop stays gap-aware.
    _gold_abs_delta = gold["logret_bar"].abs().reindex(X_traditional.index, method=None)
    _trad_gap_feature = pd.Series(0.0, index=X_traditional.index)
    _trad_times = X_traditional.index.to_list()
    for _i in range(1, len(_trad_times)):
        _t_prev, _t_curr = _trad_times[_i - 1], _trad_times[_i]
        if (_t_curr - _t_prev) > pd.Timedelta(GAP_THRESHOLD):
            _mask = (gold["logret_bar"].index > _t_prev) & (gold["logret_bar"].index <= _t_curr)
            _trad_gap_feature.iloc[_i] = float(gold["logret_bar"].loc[_mask].abs().sum())
    X_traditional["trad_movement_during_break"] = _trad_gap_feature.values

    print(f"✅ Trad daily-gap features added.  post-break bars: {int(_trad_is_post_break.sum())}  |  "
          f"non-zero 'trad_movement_during_break': {int((_trad_gap_feature != 0).sum())}")

# Column groups for downstream PCA masking
trad_ar_cols   = [c for c in X_traditional.columns if c.startswith("trad_gold_lag") or c.startswith("trad_gold_ma") or c.startswith("trad_gold_std") or c in ("is_post_break", "trad_movement_during_break")]
trad_pred_cols = [c for c in X_traditional.columns if c not in trad_ar_cols]

print(f"X_traditional shape : {X_traditional.shape}")
print(f"y_traditional shape : {y_traditional.shape}")
print(f"  predictor cols    : {len(trad_pred_cols)}  (will go through PCA if enabled)")
print(f"  AR lag cols       : {len(trad_ar_cols)}   (kept raw, bypass PCA)")
print(f"  Features preview  : {list(X_traditional.columns[:8])}")


✅ Trad daily-gap features added.  post-break bars: 19  |  non-zero 'trad_movement_during_break': 19
X_traditional shape : (7134, 31)
y_traditional shape : (7134,)
  predictor cols    : 18  (will go through PCA if enabled)
  AR lag cols       : 13   (kept raw, bypass PCA)
  Features preview  : ['crude_oil_wti_futures_price_logret', 'crude_oil_brent_futures_price_logret', 's_p_500_index_logret', 'usd_index_logret', 'vix_index_logret', 'usgg10yr_index_logret', 'eurusd_logret', 'gc1_comdty_logret']


## 5.1 Dataset registry

`DATASET_SPECS` is the single place that registers every `(X, y)` pair the training loop should iterate over.  Adding a new feature set (e.g. combined polymarket + traditional) is a one-line addition here.

| Key | X | y | PCA | Non-PCA cols |
|-----|---|---|-----|--------------|
| `poly` | Polymarket + AR | gold log-return | `PCA_N_COMPONENTS` | `ar_cols` |
| `trad` | Bloomberg traditional + AR | gold log-return | `PCA_N_COMPONENTS_TRADITIONAL` | `trad_ar_cols` |

In [13]:
# %% ── CELL 5.1 : DATASET REGISTRY ──────────────────────────────────────────
# Each entry:
#   "X"               : predictor DataFrame (already stationary, aligned to y)
#   "y"               : target Series
#   "pca_n_components": number of PCA components (0 = no PCA)
#   "non_pca_cols"    : columns that skip PCA and are fed raw to the model
#                       (AR lags, gap features, staleness counters)
DATASET_SPECS = {
    "poly": {
        "X"               : X,
        "y"               : y,
        "pca_n_components": PCA_N_COMPONENTS,
        "non_pca_cols"    : ar_cols,          # global AR cols defined in Cell 4.1
    },
    "trad": {
        "X"               : X_traditional,
        "y"               : y_traditional,
        "pca_n_components": PCA_N_COMPONENTS_TRADITIONAL,
        "non_pca_cols"    : trad_ar_cols,     # trad_gold_lag* columns
    },
}

for tag, ds in DATASET_SPECS.items():
    print(f"  [{tag}]  X={ds['X'].shape}  y={ds['y'].shape}  "
          f"pca_n={ds['pca_n_components']}  non_pca={len(ds['non_pca_cols'])} cols")


  [poly]  X=(7134, 163)  y=(7134,)  pca_n=None  non_pca=13 cols
  [trad]  X=(7134, 31)  y=(7134,)  pca_n=None  non_pca=13 cols


## 6. Modelling framework (walk-forward)

Everything below is organised so that **adding / removing a model is a one-line change**:

```python
MODEL_REGISTRY = {
    "linear": make_linear,
    "rf":     make_rf,
    "lstm":   make_lstm,
}
```

A *model factory* returns an object exposing `.fit(X,y)` and `.predict(X)`. The walk-forward loop is identical for all of them; only LSTM uses the **batched retraining** path (one refit every `LSTM_TEST_BLOCK` obs).

### Cross-validation strategy
For each `(model, window_scheme)` combo we run an **expanding or fixed-size walk-forward** where at each step the model is trained on `[t-window, t-1]` (fixed) or `[start, t-1]` (expanding) and predicts `y_t`. This is the closest scikit-learn analogue to your "train on rolling window, test on training_size+1" spec. For LSTM we predict 12 steps in one go before refitting — same walk-forward, just coarser grid.

In [14]:
# %% ── CELL 6 : HELPERS — WINDOW INDICES & METRICS ──────────────────────────
def walk_forward_indices(n: int, kind: str, size: int, step: int = 1):
    '''Yield (train_idx_slice, test_idx_slice) tuples over 0..n-1.
       kind='fixed'      -> train = [i-size : i]
       kind='expanding'  -> train = [0      : i]
       test is [i : i+step].'''
    assert kind in ("fixed","expanding")
    start = size                         # first index for which we have enough history
    for i in range(start, n, step):
        if kind == "fixed":
            tr = slice(i-size, i)
        else:
            tr = slice(0, i)
        te = slice(i, min(i+step, n))
        if te.stop <= te.start:
            break
        yield tr, te

def compute_metrics(y_true, y_pred) -> dict:
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    m = {
        "n"       : int(len(y_true)),
        "rmse"    : float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae"     : float(mean_absolute_error(y_true, y_pred)),
        "r2"      : float(r2_score(y_true, y_pred)) if len(y_true) > 1 else float("nan"),
        "dir_acc" : float(np.mean(np.sign(y_true) == np.sign(y_pred))),
        "hit_rate_nonzero" : float(np.mean((y_true != 0) & (np.sign(y_true) == np.sign(y_pred)))
                                   / max((y_true != 0).mean(), 1e-9)),
    }
    return m


In [15]:
# %% ── CELL 6.1 : MODEL FACTORIES ────────────────────────────────────────────
# ─── FIX (Operation Fixing Terrible Performance, point 2) ────────────────────
# * Ridge regularisation is now configurable via RIDGE_ALPHA.
# * Optional PCA pre-compression of the Polymarket block (NOT the AR block)
#   is added. PCA is fit on the TRAINING slice only inside each walk-forward
#   window, so there is no look-ahead leakage. Components are controlled by
#   PCA_N_COMPONENTS (set to 0/None to disable).
# * LSTM is KEPT (per user instruction). We are aware it's under-determined
#   with 161 features and 120-row windows — we track its performance as we
#   iterate on feature engineering.
# ─── UPDATE (Operation Align Traditional Predictors) ─────────────────────────
# * _pca_mask_for now accepts an explicit `non_pca_cols` list so it works for
#   both the polymarket block (ar_cols) and the traditional block (trad_ar_cols).
# * make_linear accepts `pca_n` and `non_pca_cols` kwargs so the training loop
#   can pass dataset-specific PCA settings without touching the global config.

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as _SKPipeline
from sklearn.linear_model import Ridge as _Ridge, LassoCV as _LassoCV
from sklearn.ensemble import RandomForestRegressor as _RFR
from sklearn.cross_decomposition import PLSRegression as _PLS
from sklearn.model_selection import TimeSeriesSplit as _TSS
from sklearn.feature_selection import VarianceThreshold as _VT

# Ceiling for PLS n_components inside walk-forward folds. Above this, fixed-window
# fits become numerically unstable more often; the actual operating value comes
# from the FE JSON artefact loaded earlier.
MAX_SAFE_PLS_COMPONENTS = 15

class ScaledRegressor:
    """
    Wrapper that:
      1) standardises X (and optionally y);
      2) OPTIONALLY applies PCA to a subset of columns (Polymarket block),
         leaving the remaining columns (AR block) untouched;
      3) delegates fit/predict to a core estimator.
    Keeps the main walk-forward loop model-agnostic.
    """
    def __init__(self, core, scale_y=False,
                 pca_n_components=None, pca_col_mask=None):
        self.core          = core
        self.scale_y       = scale_y
        self.xs            = StandardScaler()
        self.ys            = StandardScaler() if scale_y else None
        # --- PCA on a subset of columns (if enabled) ---
        self.pca_n         = pca_n_components if (pca_n_components and pca_n_components > 0) else None
        self.pca_col_mask  = np.asarray(pca_col_mask) if (pca_col_mask is not None and self.pca_n) else None
        self.pca           = None

    def _transform_X(self, Xs, fit: bool):
        """Split into (PCA-block, passthrough-block), apply PCA to the former."""
        if self.pca_col_mask is None:
            return Xs
        pca_part  = Xs[:, self.pca_col_mask]
        rest_part = Xs[:, ~self.pca_col_mask]
        n_comp = min(self.pca_n, pca_part.shape[1], pca_part.shape[0])
        if n_comp < 1:
            return Xs
        if fit:
            self.pca = PCA(n_components=n_comp, random_state=RANDOM_STATE)
            pca_out  = self.pca.fit_transform(pca_part)
        else:
            pca_out  = self.pca.transform(pca_part)
        return np.hstack([pca_out, rest_part])

    def fit(self, X, y):
        Xs = self.xs.fit_transform(X)
        Xs = self._transform_X(Xs, fit=True)
        if self.ys is not None:
            ys = self.ys.fit_transform(np.asarray(y).reshape(-1,1)).ravel()
            self.core.fit(Xs, ys)
        else:
            self.core.fit(Xs, y)
        return self

    def predict(self, X):
        Xs = self.xs.transform(X)
        Xs = self._transform_X(Xs, fit=False)
        pred = self.core.predict(Xs)
        if self.ys is not None:
            pred = self.ys.inverse_transform(np.asarray(pred).reshape(-1,1)).ravel()
        return np.asarray(pred).ravel()


def _pca_mask_for(columns, non_pca_cols=None) -> np.ndarray:
    """Boolean mask: True for columns that go through PCA.

    `non_pca_cols` — columns that bypass PCA and are kept raw (AR lags, gap
    features, staleness counters).  Defaults to the global `ar_cols` so the
    polymarket path is unchanged.
    """
    _exclude = set(non_pca_cols) if non_pca_cols is not None else set(ar_cols)
    return np.asarray([c not in _exclude for c in columns])


def make_linear(feature_columns=None, pca_n=None, non_pca_cols=None):
    """Ridge + optional PCA.

    `pca_n`        — override PCA_N_COMPONENTS for this call (used by the
                     dataset registry loop to pass PCA_N_COMPONENTS_TRADITIONAL
                     for the traditional block).
    `non_pca_cols` — columns that bypass PCA; defaults to global `ar_cols`.
    """
    mask   = _pca_mask_for(feature_columns, non_pca_cols=non_pca_cols) if feature_columns is not None else None
    _pca_n = pca_n if pca_n is not None else PCA_N_COMPONENTS
    return ScaledRegressor(
        Ridge(alpha=RIDGE_ALPHA, random_state=RANDOM_STATE),
        pca_n_components = _pca_n,
        pca_col_mask     = mask,
    )

def make_rf(feature_columns=None, pca_n=None, non_pca_cols=None):
    # Trees are scale-invariant — no wrapper, no PCA needed.
    # Extra kwargs accepted to keep factory signatures uniform.
    return RandomForestRegressor(
        n_estimators=RF_N_ESTIMATORS, max_depth=None,
        min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE,
    )

def _build_lstm(n_features: int, seq_len: int = LSTM_SEQ_LEN):
    model = Sequential([
        Input(shape=(seq_len, n_features)),
        LSTM(32, return_sequences=False),
        Dropout(0.2),
        Dense(16, activation="relu"),
        Dense(1, activation="linear"),
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

class LSTMRegressor:
    """LSTM with internal X/y scaling and a sliding-window transformer.
       predict(X) expects a 2-D array where row i corresponds to the "current" timestep;
       the model looks at the previous seq_len rows internally.

       Note: kept intentionally WITHOUT PCA — the LSTM already benefits from
       the raw temporal structure, and compressing features before the sequence
       window would destroy that. We track its performance as feature
       engineering progresses.
    """
    def __init__(self, seq_len=LSTM_SEQ_LEN, epochs=LSTM_EPOCHS, batch=LSTM_BATCH):
        self.seq_len, self.epochs, self.batch = seq_len, epochs, batch
        self.xs, self.ys = StandardScaler(), StandardScaler()
        self.model = None
        self._last_train_X = None   # used to build sequences that span the train→test boundary
    def _seq(self, Xs, ys=None):
        X_out, y_out = [], []
        for i in range(self.seq_len, len(Xs)):
            X_out.append(Xs[i-self.seq_len:i])
            if ys is not None:
                y_out.append(ys[i])
        X_out = np.asarray(X_out)
        return (X_out, np.asarray(y_out)) if ys is not None else X_out
    def fit(self, X, y):
        X = np.asarray(X); y = np.asarray(y).reshape(-1,1)
        Xs = self.xs.fit_transform(X)
        ys = self.ys.fit_transform(y).ravel()
        Xseq, yseq = self._seq(Xs, ys)
        self.model = _build_lstm(X.shape[1], self.seq_len)
        es = EarlyStopping(patience=3, restore_best_weights=True, monitor="loss")
        self.model.fit(Xseq, yseq, epochs=self.epochs, batch_size=self.batch,
                       verbose=0, callbacks=[es])
        self._last_train_X = Xs[-self.seq_len:]
        return self
    def predict(self, X):
        X = np.asarray(X)
        Xs = self.xs.transform(X)
        Xs_ext = np.vstack([self._last_train_X, Xs]) if self._last_train_X is not None else Xs
        preds = []
        for i in range(self.seq_len, len(Xs_ext)):
            window = Xs_ext[i-self.seq_len:i][None, ...]
            preds.append(self.model.predict(window, verbose=0)[0,0])
        preds = np.asarray(preds)
        return self.ys.inverse_transform(preds.reshape(-1,1)).ravel()

def make_lstm(feature_columns=None, pca_n=None, non_pca_cols=None):
    # Extra kwargs accepted to keep factory signatures uniform.
    if not _TF_AVAILABLE:
        return None
    return LSTMRegressor()

# ─── PLS-variant factories (FE_INPUT_MODE == "raw_panel") ────────────────────

def make_pls_preprocessor(n_components: int, poly_cols: list, ar_cols_idx: list):
    """ColumnTransformer that applies PLS only to polymarket columns and passes
    AR columns through unchanged.  Output column order: [PLS_components | AR_features].

    scale=False on PLSRegression because the StandardScaler in the pipeline is
    the only scaler; disabling the internal one prevents double-scaling.

    VarianceThreshold(threshold=1e-8) drops constant/near-constant polymarket
    columns before scaling — prevents StandardScaler blowup on early walk-forward
    windows where late-listing markets are still all-zero (ffill-filled).
    n_components is clamped to MAX_SAFE_PLS_COMPONENTS as an additional safeguard.
    """
    _safe_n = min(n_components, MAX_SAFE_PLS_COMPONENTS)
    return ColumnTransformer([
        ("poly_pls", _SKPipeline([
            ("vt",     _VT(threshold=1e-8)),
            ("scaler", StandardScaler()),
            ("pls",    _PLS(n_components=_safe_n, max_iter=500, scale=False)),
        ]), poly_cols),
        ("ar_pass", "passthrough", ar_cols_idx),
    ])


def make_pls_ridge(n_components: int, poly_cols: list, ar_cols_idx: list):
    """Linear model: PLS components (polymarket) + AR features → Ridge readout."""
    return _SKPipeline([
        ("preproc", make_pls_preprocessor(n_components, poly_cols, ar_cols_idx)),
        ("readout", _Ridge(alpha=RIDGE_ALPHA, random_state=RANDOM_STATE)),
    ])


def make_lasso_cv():
    """LassoCV operating on the full feature set (polymarket + AR).
    AR features are retained iff Lasso assigns them a non-zero coefficient."""
    return _SKPipeline([
        ("scaler", StandardScaler()),
        ("lasso",  _LassoCV(
            cv           = _TSS(n_splits=3),
            n_alphas     = 20,
            max_iter     = 10_000,
            random_state = RANDOM_STATE,
        )),
    ])


def make_rf_pls(n_components: int, poly_cols: list, ar_cols_idx: list):
    """Random Forest on PLS-compressed polymarket features + raw AR features."""
    return _SKPipeline([
        ("preproc", make_pls_preprocessor(n_components, poly_cols, ar_cols_idx)),
        ("rf",      _RFR(
            n_estimators    = RF_N_ESTIMATORS,
            max_depth       = None,
            min_samples_leaf= 5,
            n_jobs          = -1,
            random_state    = RANDOM_STATE,
        )),
    ])


class LSTMPLSWrapper:
    """PLS-preprocessed LSTM.  ColumnTransformer keeps AR features untouched.

    Custom class needed because Keras LSTMRegressor does not slot into a
    sklearn Pipeline directly (non-standard fit/predict interface for sequences).
    """
    def __init__(self, n_components: int, poly_cols: list, ar_cols_idx: list,
                 seq_len=LSTM_SEQ_LEN, epochs=LSTM_EPOCHS, batch=LSTM_BATCH):
        self.n_components = n_components
        self.poly_cols    = poly_cols
        self.ar_cols_idx  = ar_cols_idx
        self.seq_len      = seq_len
        self.epochs       = epochs
        self.batch        = batch
        self.transformer  = None
        self.core         = None

    def fit(self, X, y):
        self.transformer = make_pls_preprocessor(
            self.n_components, self.poly_cols, self.ar_cols_idx
        )
        Z = self.transformer.fit_transform(X, y)
        self.core = LSTMRegressor(
            seq_len=self.seq_len,
            epochs=self.epochs,
            batch=self.batch,
        )
        self.core.fit(Z, y)
        return self

    def predict(self, X):
        Z = self.transformer.transform(X)
        return self.core.predict(Z)


# ─── Base model registry (always registered) ────────────────────────────────
MODEL_REGISTRY = {
    "linear": make_linear,
    "rf":     make_rf,
    "lstm":   make_lstm,
}

# ─── PLS-variant models (registered only when FE_INPUT_MODE == "raw_panel") ──
# These models apply PLS to the polymarket block inside every walk-forward
# training window (no leakage).  AR columns bypass PLS via ColumnTransformer.
# "raw_panel" → add PLS variants and keep raw models for raw-vs-PLS comparison.
# "preprocessed" → input is already low-dimensional; adding PLS would double-reduce.
if FE_INPUT_MODE == "raw_panel":
    MODEL_REGISTRY.update({
        "pls_ridge": lambda: make_pls_ridge(PLS_N_COMPONENTS, POLY_COLS, AR_COLS),
        "lasso_cv":  lambda: make_lasso_cv(),
        "rf_pls":    lambda: make_rf_pls(PLS_N_COMPONENTS, POLY_COLS, AR_COLS),
        "lstm_pls":  lambda: (LSTMPLSWrapper(PLS_N_COMPONENTS, POLY_COLS, AR_COLS)
                              if _TF_AVAILABLE else None),
    })
    print("✅ PLS-variant models registered (FE_INPUT_MODE='raw_panel').")
elif FE_INPUT_MODE == "preprocessed":
    # Input is already engineered; do NOT add PLS variants (would double-reduce).
    # The existing raw-feature models ("linear", "rf", "lstm") operate on the
#    preprocessed input directly.  LassoCV may still be useful for sparsity.
    MODEL_REGISTRY.update({"lasso_cv": lambda: make_lasso_cv()})
    print("✅ preprocessed mode: base models + lasso_cv registered (PLS variants skipped).")
else:
    raise ValueError(f"Unknown FE_INPUT_MODE: {FE_INPUT_MODE!r}")

print(f"Registered models: {list(MODEL_REGISTRY.keys())}")


✅ PLS-variant models registered (FE_INPUT_MODE='raw_panel').
Registered models: ['linear', 'rf', 'lstm', 'pls_ridge', 'lasso_cv', 'rf_pls', 'lstm_pls']


In [16]:
# %% ── DIAGNOSTIC : StandardScaler blowup check ─────────────────────────────
# WHY THIS CELL EXISTS:
#   Many polymarket columns are constant (all-zero) in early walk-forward
#   windows because late-listing markets are ffill-filled with 0.0 before
#   they started trading.  StandardScaler divides by std; near-zero std
#   produces values ~1e15, which PLSRegression amplifies to inf, causing
#   rf_pls / pls_ridge / lstm_pls to raise:
#       ValueError: Input X contains infinity or a value too large for float32.
#   VarianceThreshold(threshold=1e-8) inserted before StandardScaler in
#   make_pls_preprocessor removes these columns fold-by-fold.

from sklearn.preprocessing import StandardScaler as _SS_diag

# 1. Assert raw X is finite
assert np.isfinite(X.values).all(), "X contains inf/NaN — blowup is UPSTREAM"
print("✅ Raw X is fully finite.")

# 2. Examine the first fixed120 training window
_X_tr = X.iloc[:120].values
_poly_block = _X_tr[:, POLY_COLS]

_std_tr  = np.std(_poly_block, axis=0)
n_zero   = int((_std_tr < 1e-8).sum())
n_near   = int(((0 < _std_tr) & (_std_tr < 1e-4)).sum())

_scaled  = _SS_diag().fit_transform(_poly_block)
_max_abs = float(np.max(np.abs(_scaled)))

print(f"Fixed-120 window — polymarket block shape: {_poly_block.shape}")
print(f"  Constant cols (std < 1e-8)         : {n_zero}")
print(f"  Near-constant cols (0 < std < 1e-4): {n_near}")
print(f"  Max |value| after StandardScaler   : {_max_abs:.3e}")

# 3. Summary verdict
if _max_abs > 1e6:
    print("→ diagnostic confirms StandardScaler blowup (max |scaled| > 1e6)")
else:
    print("→ diagnostic does NOT confirm blowup — investigate further")


✅ Raw X is fully finite.
Fixed-120 window — polymarket block shape: (120, 150)
  Constant cols (std < 1e-8)         : 24
  Near-constant cols (0 < std < 1e-4): 2
  Max |value| after StandardScaler   : 1.091e+01
→ diagnostic does NOT confirm blowup — investigate further


### 6.2 Walk-forward runner

For `linear` and `rf` we use `step=1` (one-step-ahead, the finest grid possible). For `lstm` we use `step=LSTM_TEST_BLOCK` (=12, ~1 hour per refit) per your spec 6.1. The same helper covers both.

In [17]:
# %% ── CELL 6.2 : WALK-FORWARD RUNNER ────────────────────────────────────────
# ─── UPDATE (Operation Align Traditional Predictors) ─────────────────────────
# Added `pca_n_components` and `non_pca_cols` kwargs so the outer dataset loop
# can pass dataset-specific PCA settings.  The polymarket path is unchanged
# (those kwargs default to None, which makes each factory fall back to globals).
def walk_forward(model_name: str, scheme_name: str, X: pd.DataFrame, y: pd.Series,
                 pca_n_components: int = None, non_pca_cols=None):
    kind, size = WINDOW_SCHEMES[scheme_name]
    step = LSTM_TEST_BLOCK if model_name == "lstm" else 1
    y_true, y_pred, ts_pred = [], [], []
    train_time_s = 0.0
    n = len(X)
    iters = list(walk_forward_indices(n, kind, size, step=step))
    feature_columns = list(X.columns)
    for k, (tr, te) in enumerate(iters):
        factory = MODEL_REGISTRY[model_name]
        try:
            mdl = factory(feature_columns=feature_columns,
                          pca_n=pca_n_components,
                          non_pca_cols=non_pca_cols)
        except TypeError:
            mdl = factory()
        if mdl is None:
            return None
        _t0 = time.perf_counter()
        mdl.fit(X.iloc[tr].values, y.iloc[tr].values)
        train_time_s += time.perf_counter() - _t0
        p  = mdl.predict(X.iloc[te].values)
        y_true.extend(y.iloc[te].values.tolist())
        y_pred.extend(np.asarray(p).tolist())
        ts_pred.extend(y.iloc[te].index.tolist())
        if (k % 50 == 0):
            print(f"  [{model_name}/{scheme_name}] step {k+1}/{len(iters)} (train={tr.stop-tr.start}, test={te.stop-te.start})")
    return {
        "y_true": np.asarray(y_true),
        "y_pred": np.asarray(y_pred),
        "timestamps": ts_pred,
        "final_model": mdl,
        "train_time_s": train_time_s,
    }


## 7. Logging and model persistence

- `Results/runs_log.txt` — one human-readable row per `(model, window, data-date)` run.
- `Results/runs_log.jsonl` — the same info as JSON for reproducibility / downstream comparison across weeks.
- `Models/<MODEL>_<WINDOW>_<DATA-DATE>.(pkl|keras)` — the **final-step** fitted model (trained on the last available window), so you can reload it without rerunning the whole walk-forward.

**"Retrain only if data changed"** logic:
- We compute `sha1` of the input gold panel CSV (content, not filename) and store it inside the sidecar JSON. On the next run, if a cached model for that `(model, window, data-hash)` triple exists and `FORCE_RETRAIN is False`, we just reload.

In [18]:
# %% ── CELL 7 : PERSISTENCE HELPERS ─────────────────────────────────────────
def file_sha1(path: Path, bufsize=1<<20) -> str:
    h = hashlib.sha1()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(bufsize), b""):
            h.update(chunk)
    return h.hexdigest()

DATA_HASH = file_sha1(panel_path)
print("Data SHA1:", DATA_HASH[:12], "…")

# ─── UPDATE (Operation Align Traditional Predictors) ─────────────────────────
# `artefact_paths` now accepts `dataset_tag` ("poly" | "trad") and `pca_n` so
# that polymarket and traditional artefacts never share file names.
# Pattern: <MODEL>_<WINDOW>_<DATASET>_h<horizonMin>m_a<ridge>_p<pca>_f<nFeat>_<dataDate>
def _fmt_alpha(a: float) -> str:
    return f"{a:g}".replace(".", "p")        # e.g. 10.0 -> '10', 0.5 -> '0p5'

def artefact_paths(model_name: str, scheme: str, data_date: str, n_features: int,
                   dataset_tag: str = "poly", pca_n: int = None) -> tuple[Path, Path]:
    """Return (model_path, metadata_path) for a given run configuration.

    `dataset_tag` is embedded in the stem so polymarket ("poly") and
    traditional ("trad") artefacts never collide even when all other
    parameters are identical.
    `pca_n` overrides PCA_N_COMPONENTS in the stem (used for the traditional
    block which uses PCA_N_COMPONENTS_TRADITIONAL).
    """
    _pca = pca_n if pca_n is not None else PCA_N_COMPONENTS
    tokens = [
        model_name.upper(),
        scheme,
        dataset_tag,
        f"h{RETURN_HORIZON_MIN}m",
        f"a{_fmt_alpha(RIDGE_ALPHA)}",
        f"p{_pca or 0}",
        f"f{n_features}",
        data_date,
    ]
    stem = "_".join(tokens)
    ext = ".keras" if model_name == "lstm" else ".pkl"
    return MODELS_DIR / f"{stem}{ext}", RESULTS_DIR / f"{stem}.json"

def cached_run_valid(meta_path: Path, data_hash: str) -> bool:
    if not meta_path.exists():
        return False
    try:
        meta = json.loads(meta_path.read_text())
        return meta.get("data_sha1") == data_hash and not FORCE_RETRAIN
    except Exception:
        return False

def save_artefacts(model_name, scheme, data_date, data_hash, metrics, result,
                   X_cols, train_time_s=None, dataset_tag="poly", pca_n=None):
    """Persist model + metadata and append to the master run logs.

    `dataset_tag` and `pca_n` are forwarded to `artefact_paths` and recorded
    in the sidecar JSON so runs are fully reproducible from the artefact alone.
    """
    n_features = len(X_cols)
    _pca_n     = pca_n if pca_n is not None else PCA_N_COMPONENTS
    mdl_path, meta_path = artefact_paths(model_name, scheme, data_date, n_features,
                                          dataset_tag=dataset_tag, pca_n=_pca_n)
    mdl = result["final_model"]
    # --- save final-step fitted model ---
    if model_name == "lstm":
        mdl.model.save(mdl_path)
        joblib.dump({"xs": mdl.xs, "ys": mdl.ys, "seq_len": mdl.seq_len,
                     "last_train_X": mdl._last_train_X},
                    mdl_path.with_suffix(".scalers.pkl"))
    else:
        joblib.dump(mdl, mdl_path)
    # --- side-car metadata (expanded) ---
    meta = {
        "model"                        : model_name,
        "window"                       : scheme,
        "dataset_tag"                  : dataset_tag,
        "data_date"                    : data_date,
        "data_sha1"                    : data_hash,
        "run_utc"                      : dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
        "train_time_s"                 : round(train_time_s, 3) if train_time_s is not None else None,
        "n_features"                   : n_features,
        "feature_sample"               : X_cols[:15],
        "metrics"                      : metrics,
        "pls_n_components"             : PLS_N_COMPONENTS,
        "pls_selection_path"           : str(PLS_SELECTION_ARTEFACT_PATH),
        "pls_selection_generated_at_utc": PLS_SELECTION_METADATA.get("generated_at_utc"),
        "config"                       : {
            # ─── target / horizon ────────────────────────────────────────────
            "bar_minutes"              : BAR_MINUTES,
            "return_horizon_min"       : RETURN_HORIZON_MIN,
            "horizon_steps"            : HORIZON_STEPS,
            # ─── AR block ────────────────────────────────────────────────────
            "ar_lags"                  : AR_LAGS,
            "ar_ma_windows"            : AR_MA_WINDOWS,
            # ─── regularisation / dim-reduction ──────────────────────────────
            "ridge_alpha"              : RIDGE_ALPHA,
            "pls_n_components"         : PLS_N_COMPONENTS,
            "pls_selection_path"       : str(PLS_SELECTION_ARTEFACT_PATH),
            # pca_applied is False when PCA is disabled (pca_n = 0/None);
            # pca_n is only stored when PCA is actually applied.
            "pca_applied"              : bool(_pca_n and _pca_n > 0),
            **({"pca_n": _pca_n} if (_pca_n and _pca_n > 0) else {}),
            # ─── feature-engineering gate / prefilter ────────────────────────
            "feature_engineering_done" : FEATURE_ENGINEERING_DONE,
            "prefilter_topN"           : MAX_FEATURES_PREFILTER if not FEATURE_ENGINEERING_DONE else None,
            # ─── LSTM / RF specifics ─────────────────────────────────────────
            "lstm_seq_len"             : LSTM_SEQ_LEN,
            "lstm_block"               : LSTM_TEST_BLOCK,
            "lstm_epochs"              : LSTM_EPOCHS,
            "lstm_batch"               : LSTM_BATCH,
            "rf_estimators"            : RF_N_ESTIMATORS,
            "random_state"             : RANDOM_STATE,
            # ─── gap handling ────────────────────────────────────────────────
            "handle_daily_gap"         : HANDLE_DAILY_GAP,
            "gap_threshold"            : GAP_THRESHOLD,
            # ─── traditional-specific ────────────────────────────────────────
            "dataset_tag"              : dataset_tag,
            "trad_max_ffill_bars"      : TRADITIONAL_MAX_FFILL_BARS if dataset_tag == "trad" else None,
            "trad_use_staleness"       : TRADITIONAL_USE_STALENESS  if dataset_tag == "trad" else None,
            "trad_ar_lags"             : TRADITIONAL_AR_LAGS        if dataset_tag == "trad" else None,
        },
    }
    meta_path.write_text(json.dumps(meta, indent=2, default=str))
    # --- append to master log ---
    train_time_str = f"{train_time_s:.1f}s" if train_time_s is not None else "N/A"
    with open(RESULTS_DIR / "runs_log.txt", "a") as f:
        f.write(
            f"{meta['run_utc']}  {model_name:7s}  {scheme:10s}  {dataset_tag:5s}  "
            f"h={RETURN_HORIZON_MIN}m  alpha={RIDGE_ALPHA:g}  pca={_pca_n or 0}  "
            f"f={n_features}  data={data_date}  rmse={metrics['rmse']:.5e}  "
            f"mae={metrics['mae']:.5e}  r2={metrics['r2']:+.4f}  "
            f"dir_acc={metrics['dir_acc']:.3f}  train_time={train_time_str}\n")
    with open(RESULTS_DIR / "runs_log.jsonl", "a") as f:
        f.write(json.dumps(meta, default=str) + "\n")
    return meta_path, mdl_path


Data SHA1: 3874ae43c5ac …


## 8. Run the full grid

`MODELS × WINDOW_SCHEMES`. Cached artefacts are reused if the data hasn't changed.

In [ ]:

# %% ── CELL 8 : RUN THE GRID ─────────────────────────────────────────────────
# ─── UPDATE (Operation Align Traditional Predictors) ─────────────────────────
# Outer loop now iterates over DATASET_SPECS so the same model grid runs on
# both the polymarket block ("poly") and the traditional Bloomberg block ("trad").
# Artefact names include the dataset tag to prevent file-name collisions.
# ─── UPDATE (PLS variants) ────────────────────────────────────────────────────
# MODELS_TO_RUN now derives from MODEL_REGISTRY so any conditionally-registered
# model (pls_ridge, rf_pls, lstm_pls, lasso_cv) is included automatically.
# PLS-variant models close over POLY_COLS / AR_COLS which are integer index lists
# derived from X (polymarket).  They are skipped for other datasets ("trad") to
# avoid applying polymarket-column indices to a differently-shaped feature matrix.
_PLS_POLY_ONLY = {k for k in MODEL_REGISTRY if "pls" in k}  # pls_ridge, rf_pls, lstm_pls

MODELS_TO_RUN = list(MODEL_REGISTRY.keys())   # driven by registry; edit registry to add/remove
all_results = []

for dataset_tag, ds in DATASET_SPECS.items():
    X_ds         = ds["X"]
    y_ds         = ds["y"]
    _pca_n       = ds["pca_n_components"]
    _non_pca     = ds.get("non_pca_cols", ar_cols)
    _N_FEATURES  = X_ds.shape[1]

    print(f"\n{'='*60}")
    print(f"  Dataset: {dataset_tag}  |  X={X_ds.shape}  |  pca_n={_pca_n}")
    print(f"{'='*60}")

    for model_name in MODELS_TO_RUN:
        if "lstm" in model_name and not _TF_AVAILABLE:
            print(f"⏭  Skipping {model_name} (TensorFlow not available).")
            continue
        # PLS-variant column indices are derived from the polymarket dataset only.
        if model_name in _PLS_POLY_ONLY and dataset_tag != "poly":
            print(f"⏭  Skipping {model_name} for dataset '{dataset_tag}' "
                  f"(PLS column indices defined for poly only).")
            continue
        for scheme in WINDOW_SCHEMES:
            mdl_path, meta_path = artefact_paths(
                model_name, scheme, panel_date, _N_FEATURES,
                dataset_tag=dataset_tag, pca_n=_pca_n,
            )
            if cached_run_valid(meta_path, DATA_HASH):
                meta = json.loads(meta_path.read_text())
                train_time_str = (f"{meta['train_time_s']:.1f}s"
                                  if meta.get("train_time_s") is not None else "N/A")
                print(f"✓ Cached: {dataset_tag}/{model_name}/{scheme} — "
                      f"rmse={meta['metrics']['rmse']:.3e}  "
                      f"dir_acc={meta['metrics']['dir_acc']:.3f}  "
                      f"train_time={train_time_str}")
                all_results.append(meta)
                continue

            print(f"▶ Training {dataset_tag}/{model_name}/{scheme} …  "
                  f"(horizon={RETURN_HORIZON_MIN}m  alpha={RIDGE_ALPHA:g}  "
                  f"pca={_pca_n or 0}  features={_N_FEATURES})")
            res = walk_forward(model_name, scheme, X_ds, y_ds,
                               pca_n_components=_pca_n, non_pca_cols=_non_pca)
            if res is None:
                continue
            metrics      = compute_metrics(res["y_true"], res["y_pred"])
            train_time_s = res["train_time_s"]
            save_artefacts(
                model_name, scheme, panel_date, DATA_HASH, metrics, res,
                list(X_ds.columns), train_time_s,
                dataset_tag=dataset_tag, pca_n=_pca_n,
            )
            all_results.append({
                "model"               : model_name,
                "window"              : scheme,
                "dataset_tag"         : dataset_tag,
                "data_date"           : panel_date,
                "n_features"          : _N_FEATURES,
                "metrics"             : metrics,
                "train_time_s"        : train_time_s,
                "pls_n_components"    : PLS_N_COMPONENTS,
                "pls_selection_path"  : str(PLS_SELECTION_ARTEFACT_PATH),
                "config"              : {
                    "return_horizon_min": RETURN_HORIZON_MIN,
                    "ridge_alpha"      : RIDGE_ALPHA,
                    "pca_applied"      : bool(_pca_n and _pca_n > 0),
                    **({"pca_n": _pca_n} if (_pca_n and _pca_n > 0) else {}),
                    "pls_n_components" : PLS_N_COMPONENTS,
                    "pls_selection_path": str(PLS_SELECTION_ARTEFACT_PATH),
                },
            })
            print(f"   ✓ rmse={metrics['rmse']:.3e}  mae={metrics['mae']:.3e}  "
                  f"r2={metrics['r2']:+.4f}  dir_acc={metrics['dir_acc']:.3f}  "
                  f"train_time={train_time_s:.1f}s")



  Dataset: poly  |  X=(7134, 163)  |  pca_n=None
  Dataset: poly  |  X=(3670, 163)  |  pca_n=30
▶ Training poly/linear/fixed120 …  (horizon=60m  alpha=10  pca=30  features=163)
  [linear/fixed120] step 1/3550 (train=120, test=1)
  [linear/fixed120] step 51/3550 (train=120, test=1)
  [linear/fixed120] step 101/3550 (train=120, test=1)
  [linear/fixed120] step 151/3550 (train=120, test=1)
  [linear/fixed120] step 201/3550 (train=120, test=1)
  [linear/fixed120] step 251/3550 (train=120, test=1)
  [linear/fixed120] step 301/3550 (train=120, test=1)
  [linear/fixed120] step 351/3550 (train=120, test=1)
  [linear/fixed120] step 401/3550 (train=120, test=1)
  [linear/fixed120] step 451/3550 (train=120, test=1)
  [linear/fixed120] step 501/3550 (train=120, test=1)
  [linear/fixed120] step 551/3550 (train=120, test=1)
  [linear/fixed120] step 601/3550 (train=120, test=1)
  [linear/fixed120] step 651/3550 (train=120, test=1)
  [linear/fixed120] step 701/3550 (train=120, test=1)
  [linear/fixed

## 9. Summary table

In [ ]:
# %% ── CELL 9 : SUMMARY ─────────────────────────────────────────────────────
import importlib, sys
sys.path.insert(0, "./Functions")
import save_summary_table as _sst
importlib.reload(_sst)
from save_summary_table import build_summary_table

summary = build_summary_table(
    all_results, RETURN_HORIZON_MIN, RIDGE_ALPHA, PCA_N_COMPONENTS,
    panel_date, RESULTS_DIR
)
summary["pls_n_components"] = PLS_N_COMPONENTS
summary["pls_selection_path"] = str(PLS_SELECTION_ARTEFACT_PATH)
summary["pls_selection_generated_at_utc"] = PLS_SELECTION_METADATA.get("generated_at_utc")

_summary_snap_name = (
    f"summary_h{RETURN_HORIZON_MIN}m_a{str(RIDGE_ALPHA).replace('.', 'p')}"
    f"_p{PCA_N_COMPONENTS or 0}_{panel_date}.csv"
)
summary.to_csv(RESULTS_DIR / _summary_snap_name, index=False)

print(f"Loaded PLS_N_COMPONENTS recorded in summary: {PLS_N_COMPONENTS}")
print(f"PLS selection artefact path           : {PLS_SELECTION_ARTEFACT_PATH}")


dataset  model    window       date  horizon_min  ridge_alpha  pca_applied  pca_n  n_features     rmse      mae             r2  dir_acc    n  train_time_s
   poly linear expanding 2026-04-19           60         10.0         True     30         163 1.715354 0.035684 -203871.098101 0.512113 3550        43.907
   poly linear  fixed120 2026-04-19           60         10.0         True     30         163 0.798999 0.025359  -44231.614567 0.550704 3550        17.998
   poly linear  fixed240 2026-04-19           60         10.0         True     30         163 1.310053 0.028431 -123635.155601 0.535569 3430        21.158
   poly linear  fixed300 2026-04-19           60         10.0         True     30         163 1.382190 0.030148 -139249.768148 0.532047 3370        21.800
   poly   lstm expanding 2026-04-19           60         10.0         True     30         163 0.004173 0.002959      -0.206826 0.573521 3550      1366.150
   poly   lstm  fixed120 2026-04-19           60         10.0         

In [ ]:
# %% ── CELL 9.1 : PLS SENSITIVITY SWEEP ──────────────────────────────────────
sensitivity_results = pd.DataFrame(
    columns=["model", "n_components", "scheme", "rmse", "mae", "r2", "dir_acc", "train_time_s"]
)

if not RUN_PLS_SENSITIVITY_SWEEP:
    print("PLS sensitivity sweep skipped (RUN_PLS_SENSITIVITY_SWEEP=False).")
elif FE_INPUT_MODE != "raw_panel":
    print(
        "PLS sensitivity sweep skipped because FE_INPUT_MODE != 'raw_panel' "
        f"(got {FE_INPUT_MODE!r})."
    )
else:
    sensitivity_grid = []
    for n in PLS_SENSITIVITY_GRID:
        n = int(n)
        if n < 1 or n == PLS_N_COMPONENTS or n in sensitivity_grid:
            continue
        sensitivity_grid.append(n)

    if not sensitivity_grid:
        print(
            "PLS sensitivity sweep skipped because no alternate component counts "
            f"remain after excluding the primary value ({PLS_N_COMPONENTS})."
        )
    else:
        poly_spec = DATASET_SPECS["poly"]
        X_sens = poly_spec["X"]
        y_sens = poly_spec["y"]
        _sens_non_pca = poly_spec.get("non_pca_cols", ar_cols)

        sensitivity_registry = {}
        for n in sensitivity_grid:
            sensitivity_registry[f"pls_ridge_n{n}"] = (
                lambda n=n: make_pls_ridge(n, POLY_COLS, AR_COLS)
            )
            sensitivity_registry[f"rf_pls_n{n}"] = (
                lambda n=n: make_rf_pls(n, POLY_COLS, AR_COLS)
            )
            sensitivity_registry[f"lstm_pls_n{n}"] = (
                lambda n=n: LSTMPLSWrapper(n, POLY_COLS, AR_COLS) if _TF_AVAILABLE else None
            )

        added_model_names = list(sensitivity_registry.keys())
        MODEL_REGISTRY.update(sensitivity_registry)
        sensitivity_rows = []

        try:
            for model_name in added_model_names:
                if "lstm" in model_name and not _TF_AVAILABLE:
                    print(f"⏭  Skipping {model_name} (TensorFlow not available).")
                    continue
                n_components = int(model_name.rsplit("_n", 1)[1])
                for scheme in WINDOW_SCHEMES:
                    print(f"▶ Sensitivity run {model_name}/{scheme} …")
                    res = walk_forward(
                        model_name,
                        scheme,
                        X_sens,
                        y_sens,
                        pca_n_components=poly_spec["pca_n_components"],
                        non_pca_cols=_sens_non_pca,
                    )
                    if res is None:
                        continue
                    metrics = compute_metrics(res["y_true"], res["y_pred"])
                    sensitivity_rows.append({
                        "model": model_name,
                        "n_components": n_components,
                        "scheme": scheme,
                        "rmse": metrics["rmse"],
                        "mae": metrics["mae"],
                        "r2": metrics["r2"],
                        "dir_acc": metrics["dir_acc"],
                        "train_time_s": res["train_time_s"],
                    })
                    print(
                        f"   ✓ rmse={metrics['rmse']:.3e}  mae={metrics['mae']:.3e}  "
                        f"r2={metrics['r2']:+.4f}  dir_acc={metrics['dir_acc']:.3f}  "
                        f"train_time={res['train_time_s']:.1f}s"
                    )
        finally:
            for model_name in added_model_names:
                MODEL_REGISTRY.pop(model_name, None)

        sensitivity_results = pd.DataFrame(
            sensitivity_rows,
            columns=["model", "n_components", "scheme", "rmse", "mae", "r2", "dir_acc", "train_time_s"],
        )

        sensitivity_date = dt.datetime.now(dt.timezone.utc).date().isoformat()
        PLS_SELECTION_LOG_DIR.mkdir(parents=True, exist_ok=True)
        sensitivity_csv_path = PLS_SELECTION_LOG_DIR / f"pls_sensitivity_{sensitivity_date}.csv"
        sensitivity_results.to_csv(sensitivity_csv_path, index=False)

        print(f"Sensitivity CSV saved      : {sensitivity_csv_path}")
        if sensitivity_results.empty:
            print("Sensitivity sweep ran but produced no result rows.")
        else:
            sensitivity_summary = (
                sensitivity_results.assign(
                    model_family=sensitivity_results["model"].str.replace(r"_n\d+$", "", regex=True)
                )
                .groupby(["model_family", "n_components"], as_index=False)["rmse"]
                .mean()
                .pivot(index="model_family", columns="n_components", values="rmse")
                .sort_index(axis=0)
                .sort_index(axis=1)
            )
            print("Mean RMSE across schemes by model family and n_components:")
            print(sensitivity_summary.to_string())


PLS sensitivity sweep skipped (RUN_PLS_SENSITIVITY_SWEEP=False).


## 10. Alternative daily-gap approaches (commented out)

The **active** gap handling (Approach 2 + 3) lives in Cell 4.1 so it is applied before training. This section keeps Approach 1 (masking) and Approach 4 (per-session standardisation) commented out for easy reverting / experimentation.


In [ ]:
# %% ── CELL 10 : BREAK-HANDLING (alternative approaches kept for reference) ──
# The ACTIVE gap-handling code (Approach 2 + 3, user-selected) now lives in
# Cell 4.1 so that it is applied BEFORE training. This cell keeps the other
# two approaches commented out so you can revert / experiment later without
# rewriting anything.
# Active approach (2 + 3) is applied in Cell 4.1 when HANDLE_DAILY_GAP=True.
# The alternatives below are kept commented out for easy toggling.

# # -------- Approach 1: drop the first bar after each break > 1 hour --------
# gap_mask = X.index.to_series().diff() > pd.Timedelta(GAP_THRESHOLD)
# X = X.loc[~gap_mask]
# y = y.loc[X.index]

# # -------- Approach 4: per-session standardisation of y --------------------
# sessions = (X.index.to_series().diff() > pd.Timedelta(GAP_THRESHOLD)).cumsum()
# y = y.groupby(sessions).transform(lambda s: (s - s.mean()) / (s.std() + 1e-12))
print("Cell 10: see Cell 4.1 for the active gap-handling logic (Approach 2 + 3).")


Cell 10: see Cell 4.1 for the active gap-handling logic (Approach 2 + 3).


## 11. How to reload a model later (no retraining)

In [ ]:
# %% ── CELL 11 : MODEL RELOAD EXAMPLE ───────────────────────────────────────
# Example — load the expanding-window Linear model trained on today's data:
# mdl_path, meta_path = artefact_paths("linear", "expanding", panel_date)
# model = joblib.load(mdl_path)
# meta  = json.loads(meta_path.read_text())
# print("Reloaded:", meta["model"], meta["window"], "metrics:", meta["metrics"])
# # For LSTM:
# # from tensorflow.keras.models import load_model
# # keras_model = load_model(mdl_path)
# # scalers = joblib.load(mdl_path.with_suffix(".scalers.pkl"))


## 12. Potential issues to watch (review notes)

1. **Cache validation may be too weak for logic changes**: `cached_run_valid(...)` only checks `data_sha1` (plus `FORCE_RETRAIN`). If preprocessing/feature logic changes but the same input CSV is used, old cached artefacts can still be reused unintentionally.

2. **Feature-set identity in filenames is based on feature count only**: artefact names include `f<n_features>`, but not a hash of the exact selected columns. Different feature compositions with the same count can map to the same filename.

3. **In-notebook package installation can hurt reproducibility**: the `openpyxl` install cell mutates the environment during execution. This is convenient, but can make runs less deterministic across machines/sessions.

4. **Walk-forward retraining is computationally heavy by design**: retraining from scratch at each step (especially for RF and LSTM) can become very slow as data grows. Consider reduced retrain cadence or warm-start style alternatives if runtime becomes a bottleneck.

## 13. Potential drawbacks and things to watch

### A. Traditional indicators — alignment and stationarity

1. **Daily-to-intraday frequency mismatch**  
   Bloomberg traditional indicators are typically daily. Forward-filling to the 5-min gold clock means every 5-min bar *between* daily updates carries the same value, so the log-return / diff is **zero for ~276 consecutive bars** and spikes once at the update bar. The model may learn the *time-of-day of the update* rather than the economic signal. Consider whether to aggregate at the daily level or add a "time since update" feature (the staleness counter partially addresses this).

2. **Large staleness counters dominate raw scale**  
   A daily indicator has staleness values ranging from 0 to ~276. `StandardScaler` inside `ScaledRegressor` normalises this, but if PCA is disabled for the traditional block the Ridge regression will see extreme staleness columns vs small log-return columns. Always keep `StandardScaler` in the pipeline.

3. **First-diff NaN → zero imputation**  
   The first row of every log-return / diff series is NaN (no prior value). These are filled with `0.0`, which is technically an imputation of "no return observed". If those rows fall inside an early training window, they slightly bias the distribution. The effect is minor but worth monitoring if the window size is small relative to the number of indicator series.

4. **Stationarity is assumed, not tested**  
   `make_stationary_features` applies log-diff for positive series and simple diff for others — a heuristic, not a confirmed ADF/KPSS test. Some series (e.g. the VIX index, credit spreads) may still be non-stationary after a single difference. Run a unit-root test on `X_traditional` columns if model performance is suspicious.

---

### B. Walk-forward comparison validity

5. **Different observation counts across datasets**  
   `X` (polymarket) and `X_traditional` are aligned to the same `X.index`, but if any traditional indicator introduces additional NaN rows the sample sizes may diverge. Always check the `n` column in the summary and ensure you are comparing models on the same hold-out windows.

6. **Cache validation only uses the data CSV hash**  
   If you change `TRADITIONAL_USE_STALENESS`, `TRADITIONAL_AR_LAGS`, or `TRADITIONAL_MAX_FFILL_BARS`, the traditional feature matrix changes but the polymarket CSV hash is unchanged. **Old `trad` artefacts will be reused without retraining.** Set `FORCE_RETRAIN = True` after any feature-engineering logic change, then revert.

7. **Naming scheme uses feature count (`f<N>`) not feature content hash**  
   Two traditional runs with different `TRADITIONAL_AR_LAGS` but the same final column count map to the same file name. A hash of `X_traditional.columns.tolist()` would make this fully robust but is not implemented.

---

### C. Model-level considerations

8. **PCA components are not comparable across datasets**  
   `pca=5` from ~15 traditional indicators retains far more variance (likely >90 %) than `pca=30` from ~150 polymarket columns (likely 60–80 %). The degree of compression is very different. Check explained-variance ratios before interpreting coefficient magnitudes.

9. **LSTM is likely underpowered for traditional indicators**  
   The traditional block has very few columns and highly autocorrelated (forward-filled) inputs. An LSTM's temporal modelling adds little over Ridge when the series are already first-differenced and nearly i.i.d. between updates. RF and Ridge results will be more informative here.

10. **R² can be meaningfully negative without the model being useless**  
    Walk-forward out-of-sample R² compares against the rolling in-sample mean. Negative R² simply means the model predicts worse than the mean on that metric. Focus on `dir_acc` (directional accuracy) and `rmse` relative to the naive random-walk baseline for a practical signal assessment.

11. **No transaction-cost model**  
    All metrics assume zero bid-ask spread and instantaneous execution. Before drawing any trading conclusion, account for realistic costs (≥ 1 tick spread for gold futures, execution delay of at least 1 bar).

---

### D. Next steps to consider

- Run **Diebold–Mariano tests** between `poly` and `trad` model variants to assess whether performance differences are statistically significant.
- Add a **combined dataset** to `DATASET_SPECS` (`"poly+trad"`) that concatenates both feature sets, and compare against each subset alone.
- Implement **SHAP values** on the RF models to understand which traditional indicators actually contribute signal.
- Extend `cached_run_valid` to also hash the feature column list (`hashlib.sha1(str(sorted(X_cols)).encode()).hexdigest()`) for stronger cache integrity.

## 14. Potential unforeseen problems after integration

1. **Traditional gap feature leakage risk if future refactors change mask logic**: the gap sums are currently bounded to `(t_prev, t_curr]` and written at `t_curr`, which is leak-safe. If this window is accidentally widened in future edits (for example including `> t_curr` bars), leakage would be introduced silently.

2. **Feature-space drift across cached runs**: cache validity still hinges on `data_sha1`. If preprocessing knobs change (AR windows, gap toggles, prefilter choices) with the same panel file, stale artefacts can still be reused.

3. **Comparability shift vs historical results**: after adding trad rolling AR means/std and trad gap features, historical `trad` performance is not directly comparable to previous `trad` runs unless you force retrain and archive old baselines separately.

4. **Collinearity increase in traditional non-PCA block**: adding `trad_gold_lag*`, `trad_gold_ma*`, `trad_gold_std*`, and break dummies to the non-PCA subset can amplify multicollinearity. Ridge usually handles this, but coefficient interpretation becomes less stable.

5. **LSTM metadata compatibility**: older result JSONs may not contain `pca_applied`/`pca_n`. The summary cell now defaults safely, but downstream scripts that expect the old key `pca_n_components` may need adjustment.

## 15. Full Model

In [ ]:

# %% ── CELL 15 : COMBINED DATASET (poly + trad + AR) ────────────────────────
# Builds a single predictor matrix that stacks:
#   (1) Polymarket first-differenced features   (from X, Cell 4.1)
#   (2) Bloomberg traditional stationary features (from X_traditional, Cell 5)
#   (3) AR gold log-return features              (already in both, de-duplicated)
#
# The combined block is registered in DATASET_SPECS as "poly+trad" and the
# existing walk-forward grid in Cell 8 will pick it up automatically if you
# add "poly+trad" to MODELS_TO_RUN or re-run this cell followed by Cell 8.
#
# PCA strategy:
#   - Polymarket block  → compressed by PCA_N_COMPONENTS  (same as "poly" run)
#   - Traditional block → compressed by PCA_N_COMPONENTS_TRADITIONAL (same as "trad" run)
#   - AR / gap features → kept raw (bypass PCA), same as both individual runs
#
# To avoid a single flat PCA swamping the trad signal with Polymarket noise,
# we apply TWO independent PCAs inside the walk-forward loop via a custom
# ScaledRegressorDual wrapper defined below.

# ─── 15.0  Align indices ────────────────────────────────────────────────────
# Both X and X_traditional are already aligned to the same modelling index
# (X.index == X_traditional.index by construction in Cell 5).  Assert this.
assert X.index.equals(X_traditional.index), (
    "X and X_traditional indices don't match — re-run Cells 4-5 before Cell 15."
)

# ─── 15.1  Build the combined feature matrix ────────────────────────────────
# AR features are already present in both X and X_traditional under different
# column names (ar_ret_* vs trad_gold_*).  We keep both sets — they carry the
# same information but allow Ridge/RF to weight them independently.
# Rename traditional AR cols to avoid collision with poly AR cols.
trad_renamed = X_traditional.copy()

# Tag every traditional column with a "trad__" prefix to prevent name clashes
# when the two DataFrames are concatenated side by side.
trad_renamed.columns = [
    f"trad__{c}" if c not in ("is_post_break",) else c
    for c in trad_renamed.columns
]

# Concatenate: poly columns first, then traditional (AR cols are at the tail
# of each block so they stay grouped).
X_combined = pd.concat([X, trad_renamed], axis=1)
y_combined = y.copy()  # same target as poly block

# Drop any residual all-NaN columns that can appear if a trad indicator had
# no overlap with the poly window.
X_combined = X_combined.dropna(axis=1, how="all")
X_combined = X_combined.ffill().fillna(0.0)

print(f"X_combined shape : {X_combined.shape}")
print(f"y_combined shape : {y_combined.shape}")

# ─── 15.2  Column group masks for dual-PCA ──────────────────────────────────
# We need to know which columns belong to which block so that:
#   (a) poly_pca_mask   → polymarket columns only (exclude AR + gap + trad)
#   (b) trad_pca_mask   → traditional columns only (exclude AR + gap + poly)
#   (c) non_pca_mask    → AR + gap features (kept raw for both blocks)

combined_ar_cols = (
    [c for c in ar_cols]                     # poly AR + gap cols (from Cell 4.1)
    + [f"trad__{c}" for c in trad_ar_cols]    # trad AR + gap cols (prefixed)
)
# Remove any that didn't survive the dropna step
combined_ar_cols = [c for c in combined_ar_cols if c in X_combined.columns]

poly_feature_cols = [c for c in X.columns       if c not in combined_ar_cols]
trad_feature_cols = [c for c in trad_renamed.columns if c not in combined_ar_cols]

all_cols = list(X_combined.columns)
poly_pca_mask = np.array([c in set(poly_feature_cols) for c in all_cols])
trad_pca_mask = np.array([c in set(trad_feature_cols) for c in all_cols])
ar_mask       = np.array([c in set(combined_ar_cols)  for c in all_cols])

print(f"  poly features   : {poly_pca_mask.sum()} cols → PCA to {PCA_N_COMPONENTS} components")
print(f"  trad features   : {trad_pca_mask.sum()} cols → PCA to {PCA_N_COMPONENTS_TRADITIONAL} components")
print(f"  AR / gap (raw)  : {ar_mask.sum()} cols")

# ─── 15.3  Dual-PCA regressor wrapper ───────────────────────────────────────
class ScaledRegressorDual:
    """
    Extends ScaledRegressor to apply TWO independent PCAs:
      - one on the Polymarket block  (poly_mask, n_poly components)
      - one on the Traditional block (trad_mask, n_trad components)
      - AR / gap columns bypass both PCAs and are kept at full scale.

    This prevents a single PCA from mixing the two very different feature
    spaces and ensures each block's compression ratio matches its individual
    run for fair comparison.
    """
    def __init__(self, core, poly_mask, trad_mask,
                 n_poly=PCA_N_COMPONENTS,
                 n_trad=PCA_N_COMPONENTS_TRADITIONAL,
                 scale_y=False):
        self.core       = core
        self.poly_mask  = poly_mask
        self.trad_mask  = trad_mask
        self.n_poly     = n_poly if (n_poly and n_poly > 0) else 0
        self.n_trad     = n_trad if (n_trad and n_trad > 0) else 0
        self.xs         = StandardScaler()
        self.ys         = StandardScaler() if scale_y else None
        self.pca_poly   = None
        self.pca_trad   = None

    def _apply_dual_pca(self, Xs, fit=False):
        poly_part = Xs[:, self.poly_mask]
        trad_part = Xs[:, self.trad_mask]
        ar_part   = Xs[:, ~self.poly_mask & ~self.trad_mask]

        # Polymarket PCA
        if self.n_poly > 0 and poly_part.shape[1] > self.n_poly:
            n = min(self.n_poly, poly_part.shape[1], poly_part.shape[0])
            if fit:
                self.pca_poly = PCA(n_components=n, random_state=RANDOM_STATE)
                poly_out = self.pca_poly.fit_transform(poly_part)
            else:
                poly_out = self.pca_poly.transform(poly_part)
        else:
            poly_out = poly_part

        # Traditional PCA
        if self.n_trad > 0 and trad_part.shape[1] > self.n_trad:
            n = min(self.n_trad, trad_part.shape[1], trad_part.shape[0])
            if fit:
                self.pca_trad = PCA(n_components=n, random_state=RANDOM_STATE)
                trad_out = self.pca_trad.fit_transform(trad_part)
            else:
                trad_out = self.pca_trad.transform(trad_part)
        else:
            trad_out = trad_part

        return np.hstack([poly_out, trad_out, ar_part])

    def fit(self, X, y):
        Xs = self.xs.fit_transform(X)
        Xs = self._apply_dual_pca(Xs, fit=True)
        if self.ys is not None:
            ys = self.ys.fit_transform(np.asarray(y).reshape(-1, 1)).ravel()
            self.core.fit(Xs, ys)
        else:
            self.core.fit(Xs, y)
        return self

    def predict(self, X):
        Xs = self.xs.transform(X)
        Xs = self._apply_dual_pca(Xs, fit=False)
        pred = self.core.predict(Xs)
        if self.ys is not None:
            pred = self.ys.inverse_transform(np.asarray(pred).reshape(-1, 1)).ravel()
        return np.asarray(pred).ravel()

# ─── 15.4  Combined model factories ─────────────────────────────────────────
# These wrap the dual-PCA logic for each model type.  RF doesn't need
# scaling or PCA (scale-invariant), LSTM keeps its own internal scaler.

def make_linear_combined(feature_columns=None, pca_n=None, non_pca_cols=None):
    return ScaledRegressorDual(
        Ridge(alpha=RIDGE_ALPHA, random_state=RANDOM_STATE),
        poly_mask=poly_pca_mask,
        trad_mask=trad_pca_mask,
        n_poly=PCA_N_COMPONENTS,
        n_trad=PCA_N_COMPONENTS_TRADITIONAL,
    )

def make_rf_combined(feature_columns=None, pca_n=None, non_pca_cols=None):
    return RandomForestRegressor(
        n_estimators=RF_N_ESTIMATORS, max_depth=None,
        min_samples_leaf=5, n_jobs=-1, random_state=RANDOM_STATE,
    )

def make_lstm_combined(feature_columns=None, pca_n=None, non_pca_cols=None):
    if not _TF_AVAILABLE:
        return None
    return LSTMRegressor()

COMBINED_MODEL_REGISTRY = {
    "linear": make_linear_combined,
    "rf"    : make_rf_combined,
    "lstm"  : make_lstm_combined,
}

# ─── 15.5  Register combined dataset + run the grid ─────────────────────────
# We use the existing walk_forward() runner directly.  For Ridge, the
# dual-PCA is handled inside ScaledRegressorDual above — walk_forward()
# doesn't need to know about it because the factory returns the right object.
# For RF and LSTM, the standard factories are used (RF: no PCA needed;
# LSTM: internal scaler handles the combined feature set).

DATASET_SPECS["poly+trad"] = {
    "X"               : X_combined,
    "y"               : y_combined,
    "pca_n_components": 0,          # handled internally by dual-PCA wrapper
    "non_pca_cols"    : combined_ar_cols,
}

MODELS_TO_RUN_COMBINED = ["linear", "rf", "lstm"]

all_results_combined = []

for model_name in MODELS_TO_RUN_COMBINED:
    if model_name == "lstm" and not _TF_AVAILABLE:
        print(f"⏭  Skipping LSTM (TensorFlow not available).")
        continue
    for scheme in WINDOW_SCHEMES:
        mdl_path, meta_path = artefact_paths(
            model_name, scheme, panel_date, X_combined.shape[1],
            dataset_tag="poly+trad", pca_n=0,
        )
        if cached_run_valid(meta_path, DATA_HASH):
            meta = json.loads(meta_path.read_text())
            print(f"✓ Cached: poly+trad/{model_name}/{scheme} — "
                  f"rmse={meta['metrics']['rmse']:.3e}  "
                  f"dir_acc={meta['metrics']['dir_acc']:.3f}")
            all_results_combined.append(meta)
            continue

        # Override the model registry for this dataset so Ridge uses dual-PCA
        _orig_registry = MODEL_REGISTRY.copy()
        MODEL_REGISTRY.update(COMBINED_MODEL_REGISTRY)

        print(f"▶ Training poly+trad/{model_name}/{scheme} …  "
              f"(horizon={RETURN_HORIZON_MIN}m  features={X_combined.shape[1]})")
        res = walk_forward(
            model_name, scheme, X_combined, y_combined,
            pca_n_components=0,            # dual-PCA is inside the wrapper
            non_pca_cols=combined_ar_cols,
        )

        # Restore the original registry so Cells 8/9 are unaffected
        MODEL_REGISTRY.update(_orig_registry)

        if res is None:
            continue

        metrics = compute_metrics(res["y_true"], res["y_pred"])
        save_artefacts(
            model_name, scheme, panel_date, DATA_HASH, metrics, res,
            list(X_combined.columns), res["train_time_s"],
            dataset_tag="poly+trad", pca_n=0,
        )
        all_results_combined.append({
            "model"       : model_name,
            "window"      : scheme,
            "dataset_tag" : "poly+trad",
            "data_date"   : panel_date,
            "n_features"  : X_combined.shape[1],
            "metrics"     : metrics,
            "train_time_s": res["train_time_s"],
        })
        print(f"   ✓ rmse={metrics['rmse']:.3e}  mae={metrics['mae']:.3e}  "
              f"r2={metrics['r2']:+.4f}  dir_acc={metrics['dir_acc']:.3f}  "
              f"train_time={res['train_time_s']:.1f}s")

# ─── 15.6  Summary — combined vs individual ─────────────────────────────────
rows_combined = []
for r in all_results + all_results_combined:
    m   = r.get("metrics", r)
    rows_combined.append({
        "dataset"    : r.get("dataset_tag", "poly"),
        "model"      : r.get("model"),
        "window"     : r.get("window"),
        "rmse"       : m.get("rmse"),
        "r2"         : m.get("r2"),
        "dir_acc"    : m.get("dir_acc"),
        "n_features" : r.get("n_features"),
        "train_time_s": r.get("train_time_s"),
    })
summary_combined = (
    pd.DataFrame(rows_combined)
    .sort_values(["model", "window", "dataset"])
    .reset_index(drop=True)
)
print("\n── Combined vs Individual summary ──")
print(summary_combined.to_string(index=False))
_snap = (f"summary_combined_h{RETURN_HORIZON_MIN}m_{panel_date}.csv")
summary_combined.to_csv(RESULTS_DIR / _snap, index=False)
print(f"\n✅ Combined run complete. Summary saved to Results/{_snap}")


X_combined shape : (3670, 194)
y_combined shape : (3670,)
  poly features   : 150 cols → PCA to 30 components
  trad features   : 18 cols → PCA to 5 components
  AR / gap (raw)  : 26 cols
✓ Cached: poly+trad/linear/fixed120 — rmse=5.660e-01  dir_acc=0.632
✓ Cached: poly+trad/linear/fixed240 — rmse=1.063e+00  dir_acc=0.579
✓ Cached: poly+trad/linear/fixed300 — rmse=1.221e+00  dir_acc=0.569
✓ Cached: poly+trad/linear/expanding — rmse=1.674e+00  dir_acc=0.528
✓ Cached: poly+trad/rf/fixed120 — rmse=2.845e-03  dir_acc=0.719
✓ Cached: poly+trad/rf/fixed240 — rmse=2.831e-03  dir_acc=0.724
✓ Cached: poly+trad/rf/fixed300 — rmse=2.971e-03  dir_acc=0.703
✓ Cached: poly+trad/rf/expanding — rmse=3.264e-03  dir_acc=0.662
✓ Cached: poly+trad/lstm/fixed120 — rmse=3.977e-03  dir_acc=0.549
✓ Cached: poly+trad/lstm/fixed240 — rmse=3.710e-03  dir_acc=0.555
✓ Cached: poly+trad/lstm/fixed300 — rmse=3.789e-03  dir_acc=0.562
✓ Cached: poly+trad/lstm/expanding — rmse=3.984e-03  dir_acc=0.579

── Combined vs I

## 16. Models using respectively polymarket data, bloomberg data and AR features

In [ ]:
# %% ── CELL 5.2 : ABLATION DATASET REGISTRY ─────────────────────────────────
# Registers three additional (X, y) pairs that isolate each feature block:
#
#   "poly_only"      → Polymarket first-diff features ONLY  (no AR lags)
#   "bloomberg_only" → Bloomberg traditional features ONLY  (no AR lags)
#   "ar_only"        → AR gold log-return features ONLY     (no market data)
#
# Purpose: ablation study — quantify the marginal contribution of each block
# independently, before combining them (see "poly", "trad", "poly+trad").
#
# ── Column selection rules ────────────────────────────────────────────────────
# The existing X and X_traditional already carry AR features appended to them.
# To isolate each block cleanly we EXCLUDE the AR/gap column groups:
#   • poly_only:      X.columns  MINUS ar_cols
#   • bloomberg_only: X_traditional.columns MINUS trad_ar_cols
#   • ar_only:        ar_cols from X (these are pure gold log-return features)
#
# Note: ar_only uses the AR feature columns that are already in X. They are
# identical in X_traditional (trad_gold_lag* = ar_ret_lag* by construction),
# so we source from X for simplicity.
# ─────────────────────────────────────────────────────────────────────────────

# ── 1. poly_only ─────────────────────────────────────────────────────────────
# polymarket_cols_kept is the list of Polymarket columns that survived the
# variance prefilter in Cell 4.1 (already excludes AR / gap cols).
# We also drop the gap features (is_post_break, poly_movement_during_break)
# so the block is purely Polymarket market data.
poly_only_cols = [
    c for c in X.columns
    if c not in ar_cols                        # drop AR lags / rolling stats
    and c not in ("is_post_break",             # drop gap dummies
                  "poly_movement_during_break")
]
X_poly_only = X[poly_only_cols].copy()
y_poly_only = y.copy()

print(f"[poly_only]      X={X_poly_only.shape}  y={y_poly_only.shape}")
print(f"  sample cols: {list(X_poly_only.columns[:5])}")

# ── 2. bloomberg_only ────────────────────────────────────────────────────────
# trad_pred_cols was defined in Cell 5 as the traditional predictor columns
# that go through PCA (i.e. everything that is NOT a trad AR lag or gap col).
# We source directly from X_traditional and exclude trad_ar_cols.
bloomberg_only_cols = [
    c for c in X_traditional.columns
    if c not in trad_ar_cols                   # drop AR lags / staleness / gap
]
X_bloomberg_only = X_traditional[bloomberg_only_cols].copy()
y_bloomberg_only = y_traditional.copy()

print(f"[bloomberg_only] X={X_bloomberg_only.shape}  y={y_bloomberg_only.shape}")
print(f"  sample cols: {list(X_bloomberg_only.columns[:5])}")

# ── 3. ar_only ───────────────────────────────────────────────────────────────
# Pure AR features: lagged gold log-returns, rolling means, rolling stds,
# plus the daily gap dummies (is_post_break, poly_movement_during_break).
# These are the columns in ar_cols that actually exist in X.
ar_only_cols = [c for c in ar_cols if c in X.columns]
X_ar_only = X[ar_only_cols].copy()
y_ar_only = y.copy()

print(f"[ar_only]        X={X_ar_only.shape}  y={y_ar_only.shape}")
print(f"  cols: {list(X_ar_only.columns)}")

# ── 4. Register in DATASET_SPECS ─────────────────────────────────────────────
# PCA strategy:
#   poly_only      → same PCA as "poly"  (Polymarket block is high-dim)
#   bloomberg_only → same PCA as "trad"  (small traditional block)
#   ar_only        → NO PCA              (already low-dim return-space features)
#
# non_pca_cols is set to [] for poly_only / bloomberg_only because there are
# NO AR features in those matrices — every column goes through PCA.
# For ar_only, all columns bypass PCA (they ARE the AR features).

DATASET_SPECS["poly_only"] = {
    "X"               : X_poly_only,
    "y"               : y_poly_only,
    "pca_n_components": PCA_N_COMPONENTS,   # same compression as poly run
    "non_pca_cols"    : [],                 # no AR cols to protect — all go through PCA
}

DATASET_SPECS["bloomberg_only"] = {
    "X"               : X_bloomberg_only,
    "y"               : y_bloomberg_only,
    "pca_n_components": PCA_N_COMPONENTS_TRADITIONAL,  # same as trad run
    "non_pca_cols"    : [],                             # no AR cols — all through PCA
}

DATASET_SPECS["ar_only"] = {
    "X"               : X_ar_only,
    "y"               : y_ar_only,
    "pca_n_components": 0,                  # no PCA — features are already in return-space
    "non_pca_cols"    : ar_only_cols,       # all cols bypass PCA (they ARE AR features)
}

# ── 5. Print updated registry ────────────────────────────────────────────────
print("\nUpdated DATASET_SPECS:")
for tag, ds in DATASET_SPECS.items():
    print(f"  [{tag:15s}]  X={ds['X'].shape}  y={ds['y'].shape}  "
          f"pca_n={ds['pca_n_components']}  non_pca={len(ds['non_pca_cols'])} cols")

[poly_only]      X=(3670, 150)  y=(3670,)
  sample cols: ['volume__will_gold_gc_hit_high_5_500_by_end_of_june_614659', 'volume1mo__will_gold_gc_hit_high_5_500_by_end_of_june_614659', 'volume1mo__will_gold_gc_hit_low_3_000_by_end_of_march_57f50e', 'volume__will_gold_gc_hit_low_3_000_by_end_of_march_57f50e', 'liquidityClob__will_bitcoin_outperform_gold_in_march_2026_f9924d']
[bloomberg_only] X=(3670, 18)  y=(3670,)
  sample cols: ['crude_oil_wti_futures_price_logret', 'crude_oil_brent_futures_price_logret', 's_p_500_index_logret', 'usd_index_logret', 'vix_index_logret']
[ar_only]        X=(3670, 13)  y=(3670,)
  cols: ['ar_ret_lag1', 'ar_ret_lag2', 'ar_ret_lag3', 'ar_ret_lag6', 'ar_ret_lag12', 'ar_ret_ma3', 'ar_ret_ma6', 'ar_ret_ma12', 'ar_ret_ma36', 'ar_ret_std12', 'ar_ret_std36', 'is_post_break', 'poly_movement_during_break']

Updated DATASET_SPECS:
  [poly           ]  X=(3670, 163)  y=(3670,)  pca_n=30  non_pca=13 cols
  [trad           ]  X=(3670, 31)  y=(3670,)  pca_n=5  non_pca=13

In [ ]:
# %% ── CELL 16 : ABLATION GRID (poly_only / bloomberg_only / ar_only) ────────
#
# Runs the full model × window grid independently for each of the three
# ablation datasets registered in Cell 5.2.  Mirrors the structure of
# Cell 15 (combined run) so results are stored and logged consistently
# with the rest of the pipeline.
#
# Datasets:
#   "poly_only"      → Polymarket diff features only  (no AR lags)
#   "bloomberg_only" → Bloomberg traditional features only (no AR lags)
#   "ar_only"        → AR gold log-return features only (no market data)
#
# Config overrides (edit here to deviate from the global defaults):
#   • MODELS_TO_RUN_ABLATION  — which models to include
#   • WINDOW_SCHEMES_ABLATION — which window schemes to include
#   • All other knobs (RIDGE_ALPHA, PCA_N_COMPONENTS, etc.) are inherited
#     from Cell 0 unless overridden below.
# ─────────────────────────────────────────────────────────────────────────────

ABLATION_DATASETS   = ["poly_only", "bloomberg_only", "ar_only"]
MODELS_TO_RUN_ABLATION  = ["linear", "rf", "lstm"]   # edit freely
WINDOW_SCHEMES_ABLATION = WINDOW_SCHEMES              # same 4 schemes as main grid

# ── Optional per-dataset config overrides ────────────────────────────────────
# Each key maps to a dict of overrides applied only for that dataset.
# Supported keys: "ridge_alpha", "pca_n_components".
# Leave empty ({}) to inherit the global Cell 0 values.
ABLATION_CONFIG_OVERRIDES = {
    "poly_only"      : {},   # e.g. {"ridge_alpha": 50.0, "pca_n_components": 20}
    "bloomberg_only" : {},   # e.g. {"pca_n_components": 3}
    "ar_only"        : {},   # ar_only never uses PCA — pca_n_components ignored
}

# ─────────────────────────────────────────────────────────────────────────────

all_results_ablation = []

for dataset_tag in ABLATION_DATASETS:

    if dataset_tag not in DATASET_SPECS:
        print(f"⚠  '{dataset_tag}' not found in DATASET_SPECS — did Cell 5.2 run?")
        continue

    ds          = DATASET_SPECS[dataset_tag]
    X_ds        = ds["X"]
    y_ds        = ds["y"]
    _non_pca    = ds.get("non_pca_cols", [])

    # Apply any per-dataset config overrides
    _overrides  = ABLATION_CONFIG_OVERRIDES.get(dataset_tag, {})
    _pca_n      = _overrides.get("pca_n_components", ds["pca_n_components"])
    _alpha      = _overrides.get("ridge_alpha", RIDGE_ALPHA)
    _N_FEATURES = X_ds.shape[1]

    print(f"\n{'='*60}")
    print(f"  Ablation dataset : {dataset_tag}")
    print(f"  X={X_ds.shape}  |  pca_n={_pca_n}  |  ridge_alpha={_alpha}")
    print(f"{'='*60}")

    for model_name in MODELS_TO_RUN_ABLATION:

        if model_name == "lstm" and not _TF_AVAILABLE:
            print(f"⏭  Skipping LSTM (TensorFlow not available).")
            continue

        for scheme in WINDOW_SCHEMES_ABLATION:

            mdl_path, meta_path = artefact_paths(
                model_name, scheme, panel_date, _N_FEATURES,
                dataset_tag=dataset_tag, pca_n=_pca_n,
            )

            # ── Cache check ──────────────────────────────────────────────────
            if cached_run_valid(meta_path, DATA_HASH):
                meta = json.loads(meta_path.read_text())
                train_time_str = (f"{meta['train_time_s']:.1f}s"
                                  if meta.get("train_time_s") is not None else "N/A")
                print(f"✓ Cached: {dataset_tag}/{model_name}/{scheme} — "
                      f"rmse={meta['metrics']['rmse']:.3e}  "
                      f"dir_acc={meta['metrics']['dir_acc']:.3f}  "
                      f"train_time={train_time_str}")
                all_results_ablation.append(meta)
                continue

            # ── Walk-forward ─────────────────────────────────────────────────
            # For ablation datasets the standard MODEL_REGISTRY factories are
            # used (no dual-PCA needed — each dataset is a single block).
            # If ridge_alpha is overridden we temporarily swap it in.
            _alpha_changed = (_alpha != RIDGE_ALPHA)
            if _alpha_changed:
                _orig_alpha = RIDGE_ALPHA
                # Ridge alpha is read at factory call time via the global —
                # patch it temporarily so make_linear picks it up.
                import builtins
                _g = globals()
                _orig_RIDGE_ALPHA = _g["RIDGE_ALPHA"]
                _g["RIDGE_ALPHA"] = _alpha

            print(f"▶ Training {dataset_tag}/{model_name}/{scheme} …  "
                  f"(horizon={RETURN_HORIZON_MIN}m  alpha={_alpha}  "
                  f"pca={_pca_n}  features={_N_FEATURES})")

            res = walk_forward(
                model_name, scheme, X_ds, y_ds,
                pca_n_components=_pca_n,
                non_pca_cols=_non_pca,
            )

            # Restore RIDGE_ALPHA if it was patched
            if _alpha_changed:
                _g["RIDGE_ALPHA"] = _orig_RIDGE_ALPHA

            if res is None:
                continue

            metrics = compute_metrics(res["y_true"], res["y_pred"])

            save_artefacts(
                model_name, scheme, panel_date, DATA_HASH, metrics, res,
                list(X_ds.columns), res["train_time_s"],
                dataset_tag=dataset_tag, pca_n=_pca_n,
            )

            all_results_ablation.append({
                "model"        : model_name,
                "window"       : scheme,
                "dataset_tag"  : dataset_tag,
                "data_date"    : panel_date,
                "n_features"   : _N_FEATURES,
                "metrics"      : metrics,
                "train_time_s" : res["train_time_s"],
            })

            print(f"   ✓ rmse={metrics['rmse']:.3e}  mae={metrics['mae']:.3e}  "
                  f"r2={metrics['r2']:+.4f}  dir_acc={metrics['dir_acc']:.3f}  "
                  f"train_time={res['train_time_s']:.1f}s")

# ── Summary table ─────────────────────────────────────────────────────────────
rows_ablation = []
for r in all_results_ablation:
    m = r.get("metrics", r)
    cfg = r.get("config", {}) if isinstance(r, dict) else {}
    rows_ablation.append({
        "dataset"      : r.get("dataset_tag"),
        "model"        : r.get("model"),
        "window"       : r.get("window"),
        "horizon_min"  : cfg.get("return_horizon_min", RETURN_HORIZON_MIN),
        "ridge_alpha"  : cfg.get("ridge_alpha", RIDGE_ALPHA),
        "pca_applied"  : cfg.get("pca_applied", False),
        "n_features"   : r.get("n_features"),
        "rmse"         : m.get("rmse"),
        "mae"          : m.get("mae"),
        "r2"           : m.get("r2"),
        "dir_acc"      : m.get("dir_acc"),
        "n"            : m.get("n"),
        "train_time_s" : r.get("train_time_s"),
    })

summary_ablation = (
    pd.DataFrame(rows_ablation)
    .sort_values(["dataset", "model", "window"])
    .reset_index(drop=True)
)

print("\n── Ablation summary (poly_only / bloomberg_only / ar_only) ──")
print(summary_ablation.to_string(index=False))

_snap = (f"summary_ablation_h{RETURN_HORIZON_MIN}m"
         f"_a{str(RIDGE_ALPHA).replace('.','p')}"
         f"_p{PCA_N_COMPONENTS or 0}_{panel_date}.csv")
summary_ablation.to_csv(RESULTS_DIR / _snap, index=False)
print(f"\n✅ Ablation run complete. Summary saved to Results/{_snap}")


  Ablation dataset : poly_only
  X=(3670, 150)  |  pca_n=30  |  ridge_alpha=10.0
✓ Cached: poly_only/linear/fixed120 — rmse=1.385e+00  dir_acc=0.507  train_time=15.0s
✓ Cached: poly_only/linear/fixed240 — rmse=1.480e+00  dir_acc=0.497  train_time=19.9s
✓ Cached: poly_only/linear/fixed300 — rmse=1.516e+00  dir_acc=0.477  train_time=21.0s
✓ Cached: poly_only/linear/expanding — rmse=1.785e+00  dir_acc=0.511  train_time=41.9s
✓ Cached: poly_only/rf/fixed120 — rmse=3.921e-03  dir_acc=0.518  train_time=906.7s
✓ Cached: poly_only/rf/fixed240 — rmse=3.789e-03  dir_acc=0.515  train_time=861.8s
✓ Cached: poly_only/rf/fixed300 — rmse=3.767e-03  dir_acc=0.504  train_time=1279.2s
▶ Training poly_only/rf/expanding …  (horizon=60m  alpha=10.0  pca=30  features=150)
  [rf/expanding] step 1/3550 (train=120, test=1)
  [rf/expanding] step 51/3550 (train=170, test=1)
  [rf/expanding] step 101/3550 (train=220, test=1)
  [rf/expanding] step 151/3550 (train=270, test=1)
  [rf/expanding] step 201/3550 (train